# 02 Demand and pricing

Part 1 (lesson 5): when, what and where customers buy. Part 2 (lesson 6): pricing and unit economics.

Data: `load_clean()` from `src/load_clean.py`, 123,655 rows, 2025-09-01 to 2026-08-31.

## 1. Demand: when, what and where do customers buy?

### 1.0 Is the data the same as in notebook 01?

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets the notebook import src/
from src.load_clean import load_clean, load_tables

pio.renderers.default = "plotly_mimetype+notebook_connected"
# Extra top margin so the title and subtitle never sit on the plot.
pio.templates["fb"] = go.layout.Template(layout=dict(margin=dict(t=118, l=70, r=30, b=60), font_size=13))
pio.templates.default = "plotly_white+fb"
OUT = ROOT / "outputs"

ITEMS = ["Watermelon", "Pineapple", "Guava", "PassionFruit", "MixedBerryPremium"]
# One fixed color per item, so an item looks the same in every chart.
ITEM_COLORS = dict(zip(ITEMS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
# Seaborn "rocket" scale, as in sns.heatmap: dark = few cups, cream = many cups.
ROCKET = ["#03051a", "#35193e", "#701f57", "#ad1759", "#e13342", "#f37651", "#f6b48f", "#faebdd"]
KITCHENS = ["Pattaya_Central", "Pattaya_Beach", "BKK_Sukhumvit", "BKK_Ladprao"]
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
LAUNCH = pd.Timestamp("2026-03-01")

clean = load_clean()
platform_names = load_tables()["platform_dim"].set_index("platform_code")["platform_name"]
TOTAL_CUPS = clean["units_sold"].sum()
TOTAL_REV = clean["gross_revenue_thb"].sum()

# One item-hour can have several rows (spellings, rate codes). Sum first, then average.
hourly = (clean.groupby(["date", "hour", "kitchen", "sku", "platform_code"], as_index=False)
          [["units_sold", "gross_revenue_thb"]].sum())
hourly["day_of_week"] = hourly["date"].dt.dayofweek
hourly["platform"] = hourly["platform_code"].map(platform_names)

# Count calendar days, not days with rows, so an hour with no sale counts as 0 cups.
calendar = pd.date_range(clean["date"].min(), clean["date"].max(), freq="D")
n_weekday = pd.Series(calendar.dayofweek).value_counts().sort_index()

assert len(clean) == 123_655
assert hourly["units_sold"].sum() == TOTAL_CUPS
assert np.isclose(hourly["gross_revenue_thb"].sum(), TOTAL_REV)
assert hourly["platform"].notna().all()
assert clean.loc[clean["sku"] == "MixedBerryPremium", "date"].min() == LAUNCH, "berry rows before launch"

# Double-date promo days (RC102) lift the whole day. Trend comparisons leave them out.
RC102_DAYS = pd.DatetimeIndex(sorted(clean.loc[clean["base_code"] == "RC102", "date"].unique()))
print(f"{len(clean):,} rows -> {len(hourly):,} item-hours | {TOTAL_CUPS:,} cups | {TOTAL_REV:,.2f} THB")
print(f"{len(calendar)} days | days per weekday: {dict(zip(DAYS, n_weekday))}")
print(f"RC102 double dates: {len(RC102_DAYS)} | by weekday: "
      f"{pd.Series(RC102_DAYS.dayofweek).map(dict(enumerate(DAYS))).value_counts().to_dict()}")


def per_day(start, end, by):
    """Cups and revenue per calendar day between two dates (both included)."""
    part = hourly[hourly["date"].between(start, end)]
    n_days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    return part.groupby(by)[["units_sold", "gross_revenue_thb"]].sum() / n_days


def subtitle(title, sub):
    """Chart title: the finding (split at ';' to fit), then a grey line saying what is plotted."""
    sub_html = f"<span style='font-size:12px;color:#52514e'>{sub}</span>"
    # Fixed distance from the top edge: works in every plotly.js version.
    return dict(text=title.replace("; ", ";<br>") + "<br>" + sub_html, font_size=16,
                x=0.01, xanchor="left", y=1, yref="container", yanchor="top", pad=dict(t=36))

123,655 rows -> 123,655 item-hours | 618,123 cups | 31,146,932.57 THB
365 days | days per weekday: {'Mon': 53, 'Tue': 52, 'Wed': 52, 'Thu': 52, 'Fri': 52, 'Sat': 52, 'Sun': 52}
RC102 double dates: 12 | by weekday: {'Tue': 5, 'Sat': 3, 'Fri': 2, 'Thu': 1, 'Mon': 1}


123,655 rows = 618,123 cups, 31.15M THB, same as the end of notebook 01. 365 days: 53 Mondays, 52 of each other day.

### 1.1 When do customers buy?

In [2]:
def heat_table(df):
    """Average cups per hour for each weekday x hour. Hours with no row count as 0."""
    full = pd.MultiIndex.from_product([range(7), range(24)], names=["day_of_week", "hour"])
    cups = df.groupby(["day_of_week", "hour"])["units_sold"].sum().reindex(full, fill_value=0)
    return cups.unstack("hour").div(n_weekday, axis=0)


heat_all = heat_table(hourly)
assert np.isclose(heat_all.mul(n_weekday, axis=0).to_numpy().sum(), TOTAL_CUPS)

peak_dow, peak_hour = heat_all.stack().idxmax()
peak = heat_all.loc[peak_dow, peak_hour]
weekday_same_hour = heat_all.loc[0:4, peak_hour].mean()
# Hours as rows: 7 wide columns leave room for 3-digit numbers in each cell.
hour_rows = heat_all.T.set_axis(DAYS, axis=1).set_axis([f"{h:02d}:00" for h in range(24)], axis=0)
fig = px.imshow(hour_rows, color_continuous_scale=ROCKET, zmin=0, aspect="auto", text_auto=".0f",
                labels=dict(x="Day", y="Hour of day", color="Cups per hour"))
fig.update_traces(xgap=0, ygap=0, textfont_size=12,
                  hovertemplate="%{x} %{y}: %{z:.0f} cups an hour<extra></extra>")
fig.update_xaxes(side="bottom", tickfont_size=13)
fig.update_coloraxes(colorbar=dict(thickness=18, outlinewidth=0, title_side="right"))
fig.update_layout(height=840, width=760, plot_bgcolor="white", title=subtitle(
    f"Weekend nights sell most: {DAYS[peak_dow]} {peak_hour}:00 averages {peak:.0f} cups an hour; "
    f"{peak / weekday_same_hour:.1f}x a weekday at {peak_hour}:00",
    "Average cups per hour, all kitchens and items, Sep 2025 - Aug 2026"))
fig.show()

cups_per_weekday = hourly.groupby("day_of_week")["units_sold"].sum() / n_weekday
print("Cups per day by weekday:", dict(zip(DAYS, cups_per_weekday.round(0).astype(int))))
print(f"Quietest hour: {heat_all.stack().idxmin()[1]}:00, {heat_all.stack().min():.0f} cups an hour")

# Weekend minus weekday cups per hour, weighted by the number of each weekday.
is_wkend = heat_all.index >= 5
wkday = heat_all[~is_wkend].mul(n_weekday[~is_wkend], axis=0).sum() / n_weekday[~is_wkend].sum()
wkend = heat_all[is_wkend].mul(n_weekday[is_wkend], axis=0).sum() / n_weekday[is_wkend].sum()
gap = wkend - wkday
print(f"Weekend extra cups a day: {gap.sum():.0f} | from 21:00-23:00: {gap.loc[21:23].sum() / gap.sum():.0%} "
      f"| from 21:00-01:00: {(gap.loc[21:23].sum() + gap.loc[0:1].sum()) / gap.sum():.0%}")

Cups per day by weekday: {'Mon': 1613, 'Tue': 1707, 'Wed': 1617, 'Thu': 1629, 'Fri': 1617, 'Sat': 1848, 'Sun': 1826}
Quietest hour: 3:00, 6 cups an hour
Weekend extra cups a day: 201 | from 21:00-23:00: 73% | from 21:00-01:00: 84%


Peak: Sat and Sun 22:00, 188 cups an hour, 1.4x a weekday at 22:00. Weekdays peak at 12:00 (~140) and 22:00 (~130). Quietest 3:00-5:00, ~6 cups an hour.

Sat-Sun sell 1,826-1,848 cups a day vs 1,613-1,707 on weekdays; 73% of the extra cups come from 21:00-23:00 (84% from 21:00-01:00).

Tuesday (1,707) is above the other weekdays (~1,620): 5 of the 12 RC102 double dates fall on a Tuesday. Measure the promo lift in lesson 7.

#### 1.1.1 Do Pattaya and Bangkok peak at different hours?

In [3]:
fig = make_subplots(rows=1, cols=4, subplot_titles=KITCHENS, shared_yaxes=True, horizontal_spacing=0.03)
peaks = []
for i, k in enumerate(KITCHENS):
    part = hourly[hourly["kitchen"] == k]
    # Share of the kitchen's own week, so a small kitchen shows its pattern as clearly as a big one.
    t = heat_table(part).mul(n_weekday, axis=0)
    t = t / t.to_numpy().sum() * 100
    fig.add_trace(go.Heatmap(z=t.T.to_numpy(), x=[d[0:2] for d in DAYS], y=[f"{h:02d}:00" for h in range(24)],
                             coloraxis="coloraxis",
                             hovertemplate="%{x} %{y}<br>%{z:.2f}% of the kitchen's week<extra></extra>"),
                  row=1, col=i + 1)
    by_hour = part.groupby("hour")["units_sold"].sum() / len(calendar)
    peaks.append({"kitchen": k, "peak_hour": by_hour.idxmax(), "cups_per_hour_at_peak": round(by_hour.max(), 1),
                  "cups_per_hour_at_12": round(by_hour[12], 1), "cups_per_hour_at_22": round(by_hour[22], 1),
                  "cups_per_day": round(part["units_sold"].sum() / len(calendar))})
peaks = pd.DataFrame(peaks).set_index("kitchen")

pat = peaks.loc[KITCHENS[:2], "peak_hour"].unique()
bkk = peaks.loc[KITCHENS[2:], "peak_hour"].unique()
assert len(pat) == len(bkk) == 1, "kitchens in one city peak at different hours"
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(side="top", tickangle=0, tickfont_size=10)
fig.update_annotations(yshift=18)  # kitchen names above the day labels
fig.update_layout(height=780, plot_bgcolor="white", margin_t=170,
                  coloraxis=dict(colorscale=ROCKET, cmin=0, colorbar=dict(title="% of week", thickness=18, outlinewidth=0)),
                  title=subtitle(f"Pattaya kitchens peak at {pat[0]}:00, Bangkok kitchens at {bkk[0]}:00",
                                 "Share of each kitchen's weekly cups by weekday x hour"))
fig.show()
display(peaks)

,peak_hour,cups_per_hour_at_peak,cups_per_hour_at_12,cups_per_hour_at_22,cups_per_day
kitchen,,,,,
Pattaya_Central,22,50.7,43.1,50.7,554
Pattaya_Beach,22,43.1,36.5,43.1,470
BKK_Sukhumvit,12,37.7,37.7,33.5,419
BKK_Ladprao,12,22.8,22.8,20.0,250


**Pattaya peaks at 22:00 (Central 51, Beach 43 cups an hour); Bangkok peaks at 12:00 (Sukhumvit 38, Ladprao 23).** Each city's other peak is still in its top 3 hours.

Fits tourist nights in Pattaya and office lunch in Bangkok. Pattaya Central is the biggest kitchen, 554 cups a day; Ladprao the smallest, 250.

#### 1.1.2 Which daypart sells most in each kitchen?

Dayparts (24h clock): Morning 6-10, Midday 11-14, Afternoon 15-17, Evening 18-21, Late night 22-5. They have different lengths (3 to 8 hours), so the table also shows cups per hour.

In [4]:
DAYPARTS = {"Morning": range(6, 11), "Midday": range(11, 15), "Afternoon": range(15, 18),
            "Evening": range(18, 22), "Late night": [22, 23, 0, 1, 2, 3, 4, 5]}
hour_to_daypart = {h: d for d, hours in DAYPARTS.items() for h in hours}
assert sorted(hour_to_daypart) == list(range(24)), "each hour must be in exactly one daypart"
hourly["daypart"] = hourly["hour"].map(hour_to_daypart)

daypart_share = (pd.concat([hourly.assign(kitchen="All"), hourly])
                 .groupby(["kitchen", "daypart"], as_index=False)["units_sold"].sum())
daypart_share["hours"] = daypart_share["daypart"].map(lambda d: len(DAYPARTS[d]))
daypart_share["share_pct"] = (daypart_share["units_sold"]
                              / daypart_share.groupby("kitchen")["units_sold"].transform("sum") * 100)
daypart_share["cups_per_hour"] = daypart_share["units_sold"] / daypart_share["hours"] / len(calendar)
assert daypart_share.loc[daypart_share["kitchen"] == "All", "units_sold"].sum() == TOTAL_CUPS

share = daypart_share.pivot(index="kitchen", columns="daypart", values="share_pct")[list(DAYPARTS)]
share = share.loc[["All"] + KITCHENS]
display(share.round(1))
display(daypart_share.pivot(index="kitchen", columns="daypart", values="cups_per_hour")[list(DAYPARTS)]
        .loc[["All"] + KITCHENS].round(1))

late_pat = share.loc[KITCHENS[:2], "Late night"].mean()
late_bkk = share.loc[KITCHENS[2:], "Late night"].mean()
plot = share.drop(index="All").reset_index().melt(id_vars="kitchen", var_name="daypart", value_name="share_pct")
fig = px.bar(plot, x="daypart", y="share_pct", color="kitchen", barmode="group",
             category_orders={"kitchen": KITCHENS, "daypart": list(DAYPARTS)},
             color_discrete_sequence=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"],
             labels={"share_pct": "% of the kitchen's cups", "daypart": ""})
fig.update_traces(marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{x}: %{y:.1f}%<extra></extra>")
fig.update_layout(height=420, title=subtitle(
    f"Midday is the biggest daypart everywhere ({share.loc[KITCHENS, 'Midday'].min():.0f}-"
    f"{share.loc[KITCHENS, 'Midday'].max():.0f}%); Pattaya sells {late_pat:.0f}% late at night vs {late_bkk:.0f}% in Bangkok",
    "Share of each kitchen's cups by daypart, Sep 2025 - Aug 2026"))
fig.show()

daypart,Morning,Midday,Afternoon,Evening,Late night
kitchen,,,,,
All,11.9,26.4,17.8,23.2,20.7
Pattaya_Central,11.9,25.0,17.7,23.6,21.8
Pattaya_Beach,11.9,25.0,17.6,23.8,21.7
BKK_Sukhumvit,11.9,28.6,17.8,22.5,19.2
BKK_Ladprao,11.9,28.5,18.0,22.5,19.1


daypart,Morning,Midday,Afternoon,Evening,Late night
kitchen,,,,,
All,40.3,111.8,100.3,98.3,43.8
Pattaya_Central,13.2,34.6,32.8,32.7,15.1
Pattaya_Beach,11.2,29.4,27.6,27.9,12.7
BKK_Sukhumvit,10.0,30.0,24.9,23.5,10.0
BKK_Ladprao,6.0,17.8,15.1,14.1,6.0


Midday is the biggest daypart in every kitchen: 25% in Pattaya, 29% in Bangkok. Late night is 22% in Pattaya vs 19% in Bangkok; evening 24% vs 22.5%.

Morning is 11.9% in all 4 kitchens. Pattaya Central per hour: midday 35, afternoon 33, evening 33, late night 15 cups. Its late night gets 21.8% of cups only because it is 8 hours long.

### 1.2 Is there a monthly trend or season?

In [5]:
hourly["month"] = hourly["date"].dt.to_period("M").dt.to_timestamp()
monthly = hourly.groupby(["month", "sku"], as_index=False)[["units_sold", "gross_revenue_thb"]].sum()
monthly["days"] = monthly["month"].dt.days_in_month
monthly["cups_per_day"] = monthly["units_sold"] / monthly["days"]
assert monthly["units_sold"].sum() == TOTAL_CUPS

trend = monthly.pivot(index="month", columns="sku", values="cups_per_day")[ITEMS]
display(trend.set_axis(trend.index.strftime("%Y-%m")).round(0))

old_peak_month = trend[ITEMS[:4]].sum(axis=1).idxmax()
berry = trend["MixedBerryPremium"].dropna()
berry_drop = 1 - berry.iloc[-1] / berry.iloc[0]

old_total = trend[ITEMS[:4]].sum(axis=1)
old_kept = old_total.iloc[-1] / old_total.loc[LAUNCH]
berry_kept = berry.iloc[-1] / berry.iloc[0]
print("Drop from April:", {s: f"Jul {1 - trend.loc['2026-07-01', s] / trend.loc['2026-04-01', s]:.1%}, "
                              f"Aug {1 - trend.loc['2026-08-01', s] / trend.loc['2026-04-01', s]:.1%}"
                           for s in ITEMS[:4]})
print(f"Mar -> Aug: old items keep {old_kept:.1%}, berry keeps {berry_kept:.1%} "
      f"-> berry down {1 - berry_kept / old_kept:.0%} after the seasonal drop")
print(f"Old items Mar -> Apr: {old_total.loc['2026-04-01'] / old_total.loc[LAUNCH] - 1:+.1%}")

fig = px.line(monthly, x="month", y="cups_per_day", color="sku", markers=True,
              category_orders={"sku": ITEMS}, color_discrete_map=ITEM_COLORS,
              labels={"cups_per_day": "Cups per day", "month": "", "sku": "Item"})
fig.update_traces(line_width=2, marker_size=8,
                  hovertemplate="%{x|%b %Y}: %{y:.0f} cups a day<extra>%{fullData.name}</extra>")
for x0, x1 in [("2025-08-16", "2025-10-16"), ("2026-05-16", "2026-08-16")]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor="#86b6ef", opacity=0.15, line_width=0)
for x in ["2025-09-16", "2026-07-01"]:
    fig.add_annotation(x=x, y=0.02, yref="paper", text="Rainy season", showarrow=False,
                       font=dict(size=11, color="#52514e"), yanchor="bottom")
fig.add_vline(x=LAUNCH, line_dash="dash", line_color="#52514e", line_width=1)
fig.add_annotation(x=LAUNCH, y=1, yref="paper", text="Berry launch", showarrow=False,
                   xanchor="right", xshift=-4, yanchor="top", font=dict(size=11, color="#52514e"))
fig.update_layout(height=450, hovermode="x unified", title=subtitle(
    f"Old items peak in {old_peak_month:%B} and are lowest in the rainy months; "
    f"berry falls {berry_drop:.0%} from launch month to August",
    "Cups per day by month (monthly cups / days in month)"))
fig.show()

sku,Watermelon,Pineapple,Guava,PassionFruit,MixedBerryPremium
month,,,,,
2025-09,550.0,365.0,228.0,181.0,NaN
2025-10,534.0,366.0,232.0,179.0,NaN
2025-11,642.0,420.0,256.0,217.0,NaN
2025-12,661.0,439.0,279.0,221.0,NaN
2026-01,663.0,444.0,284.0,227.0,NaN
2026-02,711.0,467.0,292.0,237.0,NaN
2026-03,791.0,530.0,324.0,267.0,251.0
2026-04,975.0,658.0,407.0,324.0,216.0
2026-05,849.0,542.0,344.0,271.0,168.0


Drop from April: {'Watermelon': 'Jul 44.5%, Aug 42.6%', 'Pineapple': 'Jul 45.0%, Aug 45.7%', 'Guava': 'Jul 45.0%, Aug 43.5%', 'PassionFruit': 'Jul 44.3%, Aug 42.9%'}
Mar -> Aug: old items keep 69.6%, berry keeps 32.7% -> berry down 53% after the seasonal drop
Old items Mar -> Apr: +23.6%


All 4 old items peak in April 2026 (Watermelon 975 cups a day) and are lowest in Sep-Oct 2025 and Jun-Aug 2026, the rainy season. Watermelon: 534 (Oct) -> 975 (Apr) -> 541 (Jul).

**Berry falls 67% (251 -> 82 cups a day, Mar -> Aug); old items keep 70% over the same months, so berry is down ~53% after the season.** It fell in Apr-May, the peak season: mostly novelty fade, not rain.

April jump: Songkran week. On 16 Apr every old item sold 63-70% above its baseline without a promo, and that day is the only promo day in the week (notebook 03), so the jump is the holiday, not promos.

### 1.3 Which items sell most when all 5 exist?

In [6]:
MIX_START, MIX_END = "2026-03-01", "2026-08-31"
sales_mix = per_day(MIX_START, MIX_END, "sku").reindex(ITEMS)
sales_mix.columns = ["cups_per_day", "revenue_thb_per_day"]
sales_mix["share_cups_pct"] = sales_mix["cups_per_day"] / sales_mix["cups_per_day"].sum() * 100
sales_mix["share_revenue_pct"] = sales_mix["revenue_thb_per_day"] / sales_mix["revenue_thb_per_day"].sum() * 100
sales_mix["avg_price_thb"] = sales_mix["revenue_thb_per_day"] / sales_mix["cups_per_day"]
sales_mix = sales_mix.sort_values("cups_per_day", ascending=False)
n_mix_days = (pd.Timestamp(MIX_END) - pd.Timestamp(MIX_START)).days + 1
mix_rev = hourly.loc[hourly["date"].between(MIX_START, MIX_END), "gross_revenue_thb"].sum()
assert np.isclose(sales_mix["revenue_thb_per_day"].sum() * n_mix_days, mix_rev)
display(sales_mix.round(1))

top, berry_row = sales_mix.iloc[0], sales_mix.loc["MixedBerryPremium"]
fig = px.bar(sales_mix.reset_index(), x="cups_per_day", y="sku", orientation="h", color="sku",
             color_discrete_map=ITEM_COLORS, category_orders={"sku": list(sales_mix.index)},
             text=[f"{c:.0f} cups ({s:.0f}%)" for c, s in zip(sales_mix["cups_per_day"], sales_mix["share_cups_pct"])],
             labels={"cups_per_day": "Cups per day", "sku": ""})
fig.update_traces(textposition="outside", cliponaxis=False, marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{y}: %{x:.0f} cups a day<extra></extra>")
fig.update_layout(height=360, showlegend=False, xaxis_range=[0, sales_mix["cups_per_day"].max() * 1.35],
                  title=subtitle(
                      f"Watermelon sells {top['share_cups_pct']:.0f}% of cups; berry sells {berry_row['share_cups_pct']:.0f}% "
                      f"of cups but {berry_row['share_revenue_pct']:.0f}% of gross revenue",
                      "Cups per day, Mar - Aug 2026 (all 5 items on sale). Gross revenue = after discount, before commission."))
fig.show()

,cups_per_day,revenue_thb_per_day,share_cups_pct,share_revenue_pct,avg_price_thb
sku,,,,,
Watermelon,715.6,31595.6,38.3,32.7,44.1
Pineapple,472.0,22720.1,25.2,23.5,48.1
Guava,294.2,14145.0,15.7,14.6,48.1
PassionFruit,236.0,15038.3,12.6,15.5,63.7
MixedBerryPremium,152.4,13256.1,8.1,13.7,87.0


Watermelon 716 cups a day (38%), Pineapple 472 (25%), Guava 294 (16%), PassionFruit 236 (13%), berry 152 (8%).

Berry is last in cups but 14% of gross revenue (after discount, before commission), because its average price is ~87 THB vs 44-64 THB. PassionFruit (13% of cups) also earns more than Guava (16% of cups).

### 1.4 Did Mixed Berry add new sales or take sales from old items?

In [7]:
def launch_effect(before, after, by="sku"):
    """Cups per day before vs after the launch, per group, with a TOTAL row."""
    b = per_day(*before, by)["units_sold"]
    a = per_day(*after, by)["units_sold"]
    t = pd.DataFrame({"before": b, "after": a}).fillna(0)
    t.loc["TOTAL"] = t.sum()
    t["change"] = t["after"] - t["before"]
    t["change_pct"] = (t["change"] / t["before"] * 100).where(t["before"] > 0)
    return t


WINDOWS = {
    "half_year": (("2025-09-01", "2026-02-28"), ("2026-03-01", "2026-08-31")),
    "4_weeks": (("2026-02-01", "2026-02-28"), ("2026-03-01", "2026-03-28")),
}
for name, windows in WINDOWS.items():
    for start, end in windows:
        days = pd.date_range(start, end)
        n_rc102 = days.isin(RC102_DAYS).sum()
        print(f"{name} {start} to {end}: {len(days)} days, weekdays {days.dayofweek.value_counts().sort_index().tolist()}, "
              f"RC102 days {n_rc102}")
assert [len(pd.date_range(*w)) for w in WINDOWS["4_weeks"]] == [28, 28]
assert [len(pd.date_range(*w)) for w in WINDOWS["half_year"]] == [181, 184]

effects = {name: launch_effect(b, a).reindex(ITEMS + ["TOTAL"]) for name, (b, a) in WINDOWS.items()}

# Same 4-week test on normal days only (RC102 days removed from both windows).
old_daily = (hourly[hourly["sku"] != "MixedBerryPremium"].groupby("date")["units_sold"].sum()
             .reindex(calendar, fill_value=0))
normal = old_daily[~old_daily.index.isin(RC102_DAYS)]
before_n, after_n = (normal[slice(*w)].mean() for w in WINDOWS["4_weeks"])
print(f"Old items, 4 weeks, normal days only: {before_n:,.0f} -> {after_n:,.0f} cups a day ({after_n / before_n - 1:+.1%})")
print("(a) Sep 2025 - Feb 2026 vs Mar - Aug 2026, cups per day")
display(effects["half_year"].round(1))
print("(b) 4 weeks before (Feb 1-28) vs 4 weeks after (Mar 1-28), cups per day")
display(effects["4_weeks"].round(1))

half_year 2025-09-01 to 2026-02-28: 181 days, weekdays [26, 26, 26, 26, 26, 26, 25], RC102 days 6
half_year 2026-03-01 to 2026-08-31: 184 days, weekdays [27, 26, 26, 26, 26, 26, 27], RC102 days 6
4_weeks 2026-02-01 to 2026-02-28: 28 days, weekdays [4, 4, 4, 4, 4, 4, 4], RC102 days 1
4_weeks 2026-03-01 to 2026-03-28: 28 days, weekdays [4, 4, 4, 4, 4, 4, 4], RC102 days 1


Old items, 4 weeks, normal days only: 1,668 -> 1,873 cups a day (+12.3%)
(a) Sep 2025 - Feb 2026 vs Mar - Aug 2026, cups per day


,before,after,change,change_pct
sku,,,,
Watermelon,626.0,715.6,89.6,14.3
Pineapple,416.1,472.0,55.9,13.4
Guava,261.7,294.2,32.5,12.4
PassionFruit,209.9,236.0,26.1,12.4
MixedBerryPremium,0.0,152.4,152.4,NaN
TOTAL,1513.7,1870.3,356.6,23.6


(b) 4 weeks before (Feb 1-28) vs 4 weeks after (Mar 1-28), cups per day


,before,after,change,change_pct
sku,,,,
Watermelon,711.2,796.8,85.6,12.0
Pineapple,466.8,531.3,64.5,13.8
Guava,291.9,326.5,34.7,11.9
PassionFruit,236.8,267.8,31.0,13.1
MixedBerryPremium,0.0,257.8,257.8,NaN
TOTAL,1706.6,2180.2,473.5,27.7


(a) Half-years: every old item grew 12-14%; total +357 cups a day (+24%), of which berry is 152.

(b) 4 weeks: old items +12% to +14%, berry 258 cups a day, total +474 (+28%). No old item fell in either window.

Each 4-week window has 1 RC102 day (2 Feb, 3 Mar). Without them old items still go 1,668 -> 1,873 cups a day (+12.3%).

In [8]:
# Normal days only: one RC102 day lifts a whole week. Divide by days on sale, so berry's launch week is not diluted.
weeks = hourly[hourly["date"].between("2026-01-05", "2026-04-26") & ~hourly["date"].isin(RC102_DAYS)].copy()
weeks["week"] = weeks["date"] - pd.to_timedelta(weeks["date"].dt.dayofweek, unit="D")
weeks["group"] = np.where(weeks["sku"] == "MixedBerryPremium", "MixedBerryPremium", "4 old items")
weekly = (weeks.groupby(["week", "group"], as_index=False)
          .agg(units_sold=("units_sold", "sum"), days=("date", "nunique")))
weekly["cups_per_day"] = weekly["units_sold"] / weekly["days"]

jan = normal["2026-01-05":"2026-02-01"].mean()
feb = normal["2026-02-02":"2026-02-22"].mean()
old_weeks = weekly[weekly["group"] == "4 old items"].set_index("week")["cups_per_day"]
print(f"Old items, normal days: Jan {jan:,.0f} -> Feb {feb:,.0f} cups a day ({feb / jan - 1:+.1%})")
print("Old items by week after launch:", old_weeks["2026-03-01":].round(0).astype(int).to_dict())
print("Berry by week:", weekly[weekly["group"] == "MixedBerryPremium"].set_index("week")["cups_per_day"]
      .round(0).astype(int).to_dict())

fig = px.line(weekly, x="week", y="cups_per_day", color="group", markers=True,
              color_discrete_map={"4 old items": "#2a78d6", "MixedBerryPremium": "#e87ba4"},
              labels={"cups_per_day": "Cups per day", "week": "Week starting", "group": ""})
fig.update_traces(line_width=2, marker_size=8,
                  hovertemplate="Week of %{x|%d %b}: %{y:.0f} cups a day<extra>%{fullData.name}</extra>")
fig.add_vline(x=LAUNCH, line_dash="dash", line_color="#52514e", line_width=1)
fig.add_annotation(x=LAUNCH, y=1, yref="paper", text="Berry launch", showarrow=False,
                   xanchor="left", xshift=4, yanchor="top", font=dict(size=11, color="#52514e"))
fig.update_layout(height=420, hovermode="x unified", title=subtitle(
    f"Old items rose {feb / jan - 1:.0%} before launch ({jan:,.0f} -> {feb:,.0f} a day, Jan -> Feb); "
    "they kept rising after it",
    "Cups per day by week, normal days only (RC102 double dates removed). Berry's first point = launch day."))
fig.show()

Old items, normal days: Jan 1,584 -> Feb 1,669 cups a day (+5.4%)
Old items by week after launch: {Timestamp('2026-03-02 00:00:00'): 1868, Timestamp('2026-03-09 00:00:00'): 1895, Timestamp('2026-03-16 00:00:00'): 1843, Timestamp('2026-03-23 00:00:00'): 1874, Timestamp('2026-03-30 00:00:00'): 1976, Timestamp('2026-04-06 00:00:00'): 2288, Timestamp('2026-04-13 00:00:00'): 2793, Timestamp('2026-04-20 00:00:00'): 2136}
Berry by week: {Timestamp('2026-02-23 00:00:00'): 322, Timestamp('2026-03-02 00:00:00'): 262, Timestamp('2026-03-09 00:00:00'): 269, Timestamp('2026-03-16 00:00:00'): 268, Timestamp('2026-03-23 00:00:00'): 182, Timestamp('2026-03-30 00:00:00'): 189, Timestamp('2026-04-06 00:00:00'): 228, Timestamp('2026-04-13 00:00:00'): 253, Timestamp('2026-04-20 00:00:00'): 190}


**Old items were already rising before launch: 1,584 cups a day in January -> 1,669 in February (+5.4%, normal days).** After launch: 1,843-1,895 in March weeks, 2,288 in early April, 2,793 in the Songkran week.

Old items also rose 24% from March to April with no new item. A seasonal ramp this size can hide a small loss of old-item sales.

In [9]:
def kitchen_effect(before, after):
    """Old items before/after, berry after and change %, per kitchen."""
    b = per_day(*before, ["kitchen", "sku"])["units_sold"].unstack().fillna(0)
    a = per_day(*after, ["kitchen", "sku"])["units_sold"].unstack().fillna(0)
    old = [s for s in ITEMS if s != "MixedBerryPremium"]
    t = pd.DataFrame({"old_before": b[old].sum(axis=1), "old_after": a[old].sum(axis=1),
                      "berry_after": a["MixedBerryPremium"],
                      "total_before": b.sum(axis=1), "total_after": a.sum(axis=1)})
    t["old_change_pct"] = (t["old_after"] / t["old_before"] - 1) * 100
    t["total_change_pct"] = (t["total_after"] / t["total_before"] - 1) * 100
    return t.loc[KITCHENS]


kitchen_effects = {name: kitchen_effect(b, a) for name, (b, a) in WINDOWS.items()}
for name, t in kitchen_effects.items():
    assert np.isclose(t["total_after"].sum(), effects[name].loc["TOTAL", "after"])
    print(f"(c) {name}, cups per day")
    display(t.round(1))

(c) half_year, cups per day


,old_before,old_after,berry_after,total_before,total_after,old_change_pct,total_change_pct
kitchen,,,,,,,
Pattaya_Central,495.3,562.5,50.0,495.3,612.5,13.6,23.7
Pattaya_Beach,419.0,477.3,42.3,419.0,519.6,13.9,24.0
BKK_Sukhumvit,375.7,423.9,37.5,375.7,461.4,12.8,22.8
BKK_Ladprao,223.7,254.2,22.6,223.7,276.8,13.6,23.8


(c) 4_weeks, cups per day


,old_before,old_after,berry_after,total_before,total_after,old_change_pct,total_change_pct
kitchen,,,,,,,
Pattaya_Central,560.2,625.7,85.1,560.2,710.8,11.7,26.9
Pattaya_Beach,475.0,539.0,70.5,475.0,609.5,13.5,28.3
BKK_Sukhumvit,422.9,472.1,62.4,422.9,534.5,11.6,26.4
BKK_Ladprao,248.6,285.6,39.8,248.6,325.4,14.9,30.9


(c) Same in every kitchen: old items +11.6% to +14.9% in the 4-week window, +12.8% to +13.9% over the half-years. In its first 4 weeks berry sells 40-85 cups a day per kitchen, 13-14% of old-item cups in each.

Limitation: no Mar-Aug 2025 data, and berry has the same share in every kitchen, so there is no group without berry to compare. This design cannot detect cannibalization smaller than the seasonal ramp; the safe claim is "old items did not fall".

### 1.5 Where are the sales?

In [10]:
kp = hourly.groupby(["kitchen", "platform"], as_index=False)[["units_sold", "gross_revenue_thb"]].sum()
kitchen_rev = kp.groupby("kitchen")["gross_revenue_thb"].transform("sum")
kp["kitchen_revenue_share_pct"] = kitchen_rev / TOTAL_REV * 100
kp["platform_share_in_kitchen_pct"] = kp["gross_revenue_thb"] / kitchen_rev * 100
kp["revenue_thb_per_day"] = kp["gross_revenue_thb"] / len(calendar)
kitchen_platform_share = kp.sort_values(["kitchen_revenue_share_pct", "platform"], ascending=[False, False])
assert np.isclose(kitchen_platform_share["gross_revenue_thb"].sum(), TOTAL_REV)

summary = kitchen_platform_share.pivot(index="kitchen", columns="platform", values="platform_share_in_kitchen_pct")
summary.insert(0, "kitchen_revenue_share_pct", kitchen_platform_share.groupby("kitchen")["kitchen_revenue_share_pct"].first())
summary = summary.loc[KITCHENS]
display(summary.round(1))

pattaya = summary.loc[KITCHENS[:2], "kitchen_revenue_share_pct"].sum()
lm = summary["LINE MAN"]
fig = px.bar(kitchen_platform_share, x="revenue_thb_per_day", y="kitchen", color="platform", orientation="h",
             category_orders={"kitchen": KITCHENS, "platform": ["LINE MAN", "Grab"]},
             color_discrete_map={"LINE MAN": "#2a78d6", "Grab": "#eb6834"},
             labels={"revenue_thb_per_day": "Revenue per day (THB)", "kitchen": "", "platform": ""})
fig.update_traces(marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{y}: %{x:,.0f} THB a day<extra>%{fullData.name}</extra>")
fig.update_layout(height=360, title=subtitle(
    f"Pattaya makes {pattaya:.0f}% of gross revenue; LINE MAN has {lm.min():.0f}-{lm.max():.0f}% of it in every kitchen",
    "Gross revenue per day by kitchen and platform, Sep 2025 - Aug 2026"))
fig.show()

platform,kitchen_revenue_share_pct,Grab,LINE MAN
kitchen,,,
Pattaya_Central,32.8,44.7,55.3
Pattaya_Beach,27.7,43.1,56.9
BKK_Sukhumvit,24.7,45.2,54.8
BKK_Ladprao,14.8,43.7,56.3


Pattaya makes 60% of gross revenue (Central 33%, Beach 28%); Bangkok 40% (Sukhumvit 25%, Ladprao 15%).

LINE MAN has 55-57% of gross revenue in every kitchen (56% overall), Grab 43-45%. No kitchen leans to one platform.

#### 1.5.1 Do LINE MAN and Grab peak at different hours?

In [11]:
by_hour = hourly.groupby(["hour", "platform"], as_index=False)["units_sold"].sum()
by_hour["cups_per_hour"] = by_hour["units_sold"] / len(calendar)
assert by_hour["units_sold"].sum() == TOTAL_CUPS
wide = by_hour.pivot(index="hour", columns="platform", values="cups_per_hour")
lm_share = wide["LINE MAN"] / wide.sum(axis=1) * 100
print("Peak hour:", wide.idxmax().to_dict())
print(f"LINE MAN share of cups by hour: {lm_share.min():.1f}% to {lm_share.max():.1f}%")

fig = px.line(by_hour, x="hour", y="cups_per_hour", color="platform", markers=True,
              category_orders={"platform": ["LINE MAN", "Grab"]},
              color_discrete_map={"LINE MAN": "#2a78d6", "Grab": "#eb6834"},
              labels={"cups_per_hour": "Cups per hour", "hour": "Hour of day", "platform": ""})
fig.update_traces(line_width=2, marker_size=8,
                  hovertemplate="%{x}:00: %{y:.0f} cups an hour<extra>%{fullData.name}</extra>")
fig.update_xaxes(dtick=2, tickangle=0, range=[-0.5, 23.5])
fig.update_layout(height=400, hovermode="x unified", title=subtitle(
    f"LINE MAN and Grab peak at the same hour ({wide['LINE MAN'].idxmax()}:00); "
    f"LINE MAN keeps {lm_share.min():.0f}-{lm_share.max():.0f}% of cups in every hour",
    "Average cups per hour by platform, all kitchens, Sep 2025 - Aug 2026"))
fig.show()

Peak hour: {'Grab': 22, 'LINE MAN': 22}
LINE MAN share of cups by hour: 55.0% to 56.7%


Both platforms peak at 22:00 (LINE MAN 82, Grab 66 cups an hour) and have the same daily shape; LINE MAN's share of cups stays 55.0-56.7% in every hour.

So the timing does not differ by platform. Price, promo mix and commission (LINE MAN 30% vs Grab 24%) by platform come in lesson 6.

### 1.6 What does the app need from this part?

In [12]:
OUT.mkdir(exist_ok=True)

heat_rows = []
for k in ["All"] + KITCHENS:
    part = hourly if k == "All" else hourly[hourly["kitchen"] == k]
    t = heat_table(part).stack().rename("avg_cups_per_hour").reset_index()
    t.insert(0, "kitchen", k)
    t["day_name"] = t["day_of_week"].map(dict(enumerate(DAYS)))
    heat_rows.append(t)
demand_heatmap = pd.concat(heat_rows, ignore_index=True)
# Rows for "All" kitchens or the TOTAL item repeat other rows. The app must filter is_total before summing.
demand_heatmap["is_total"] = demand_heatmap["kitchen"] == "All"
daypart_share["is_total"] = daypart_share["kitchen"] == "All"

monthly_trend = monthly[["month", "sku", "days", "units_sold", "cups_per_day"]]
launch_rows = []
for name, t in effects.items():
    launch_rows.append(t.rename_axis("sku").reset_index().assign(window=name))
berry_launch_effect = pd.concat(launch_rows, ignore_index=True)
berry_launch_effect["is_total"] = berry_launch_effect["sku"] == "TOTAL"

tables = {
    "demand_heatmap.csv": demand_heatmap,
    "daypart_share.csv": daypart_share,
    "sales_mix.csv": sales_mix.rename_axis("sku").reset_index().assign(period="2026-03 to 2026-08"),
    "kitchen_platform_share.csv": kitchen_platform_share,
    "monthly_item_trend.csv": monthly_trend,
    "berry_launch_effect.csv": berry_launch_effect,
}
for file, df in tables.items():
    df.to_csv(OUT / file, index=False, encoding="utf-8")
    print(f"{file}: {len(df)} rows")

# The saved tables must add back to the clean totals.
h = pd.read_csv(OUT / "demand_heatmap.csv")
h_all = h[h["kitchen"] == "All"].pivot(index="day_of_week", columns="hour", values="avg_cups_per_hour")
assert np.isclose(h_all.mul(n_weekday, axis=0).to_numpy().sum(), TOTAL_CUPS)
for k in KITCHENS:
    hk = h[h["kitchen"] == k].pivot(index="day_of_week", columns="hour", values="avg_cups_per_hour")
    assert np.isclose(hk.mul(n_weekday, axis=0).to_numpy().sum(), hourly.loc[hourly["kitchen"] == k, "units_sold"].sum())
d = pd.read_csv(OUT / "daypart_share.csv")
assert d.loc[d["kitchen"] == "All", "units_sold"].sum() == TOTAL_CUPS
assert np.isclose(d.groupby("kitchen")["share_pct"].sum(), 100).all()
assert d.loc[~d["is_total"], "units_sold"].sum() == TOTAL_CUPS
assert np.isclose(pd.read_csv(OUT / "kitchen_platform_share.csv")["gross_revenue_thb"].sum(), TOTAL_REV)
b = pd.read_csv(OUT / "berry_launch_effect.csv")
assert np.isclose(b[~b["is_total"]].groupby("window")["after"].sum().sort_index(),
                  b[b["is_total"]].set_index("window")["after"].sort_index()).all()
assert pd.read_csv(OUT / "monthly_item_trend.csv")["units_sold"].sum() == TOTAL_CUPS
assert np.isclose(pd.read_csv(OUT / "sales_mix.csv")["share_cups_pct"].sum(), 100)
print(f"Saved tables match clean totals: {TOTAL_CUPS:,} cups, {TOTAL_REV:,.2f} THB")

demand_heatmap.csv: 840 rows
daypart_share.csv: 25 rows
sales_mix.csv: 5 rows
kitchen_platform_share.csv: 8 rows
monthly_item_trend.csv: 54 rows
berry_launch_effect.csv: 12 rows


Saved tables match clean totals: 618,123 cups, 31,146,932.57 THB


6 tables in `outputs/`: the 4 asked for, plus `monthly_item_trend.csv` and `berry_launch_effect.csv`. Rows for kitchen "All" or sku "TOTAL" have `is_total=True`; filter them out before summing. In `kitchen_platform_share.csv`, `kitchen_revenue_share_pct` repeats on both platform rows.

## Findings -> next steps
1. Pattaya peaks at 22:00, Bangkok at 12:00; Sat-Sun 22:00 is the top slot (188 cups an hour, 1.4x a weekday) -> plan prep and staff by city and hour (lesson 10); test promos in quiet hours, not peaks (lesson 7).
2. Old items peak in April (Watermelon 975 cups a day) and are 44-45% lower in July -> start the Sep-Nov forecast from rainy-season levels; the Songkran week (2,793 old-item cups a day) is the holiday, not promos (notebook 03).
3. Berry is 8% of cups but 14% of gross revenue (avg 87 THB), and down 67% since launch (251 -> 82 cups a day, ~53% after season) -> check berry CM in lesson 6 and cut berry prep in lesson 10.
4. Old items did not fall after launch (+12-15% in every kitchen), but they were already rising (+5.4% Jan -> Feb, normal days) -> this design cannot detect cannibalization smaller than the seasonal ramp; claim "no fall", not "no loss".
5. LINE MAN has 55-57% of gross revenue in every kitchen and of cups in every hour, at 30% commission vs Grab 24% -> compare CM per cup by platform in lesson 6.

## 2. Pricing: what does one cup earn?

### 2.0 Does every order find its price and costs?

In [13]:
from src.economics import MONEY as MONEY_COLS, enrich, unit_economics

tables = load_tables()
enriched = enrich(clean, tables)
enriched["platform"] = enriched["platform_code"].map(platform_names)
print(f"{len(enriched):,} rows after joins (clean: {len(clean):,})")
print("Missing after joins:", int(enriched[["commission_rate", "discount_pct", "fruit_cost_per_cup_thb",
                                            "pack_labor_per_cup_thb", "base_price_thb"]].isna().sum().sum()))

# List price check: the price on each code / (1 - discount) vs base_price_thb.
implied = enriched["price_thb"] / (1 - enriched["discount_pct"])
gap = (implied - enriched["base_price_thb"]).abs()
print(f"Rows where price / (1 - discount) != base_price_thb: {(gap > 0.005).sum()} | largest gap: {gap.max():.4f} THB")
display(enriched.groupby(["sku", "base_code"])["price_thb"].first().unstack().reindex(ITEMS))

launch = enriched[(enriched["sku"] == "MixedBerryPremium") & (enriched["date"] == LAUNCH)]
print(f"Berry launch day: {len(launch)} rows, fruit cost {launch['fruit_cost_per_cup_thb'].unique()} THB a cup "
      f"(week 2026-03-02 cost, copied into week 2026-02-23)")

enriched.to_parquet(ROOT / "data" / "processed" / "orders_enriched.parquet", index=False)
saved = pd.read_parquet(ROOT / "data" / "processed" / "orders_enriched.parquet")
assert saved.shape == enriched.shape and np.isclose(saved["cm_thb"].sum(), enriched["cm_thb"].sum())
print(f"orders_enriched.parquet: {saved.shape[0]:,} rows x {saved.shape[1]} columns")

123,655 rows after joins (clean: 123,655)
Missing after joins: 0
Rows where price / (1 - discount) != base_price_thb: 0 | largest gap: 0.0000 THB


base_code,RC000,RC101,RC102,RC103
sku,,,,
Watermelon,45.0,38.25,36.0,39.60
Pineapple,49.0,41.65,39.2,43.12
Guava,49.0,41.65,39.2,43.12
PassionFruit,65.0,55.25,52.0,57.20
MixedBerryPremium,89.0,75.65,71.2,78.32


Berry launch day: 75 rows, fruit cost [33.69] THB a cup (week 2026-03-02 cost, copied into week 2026-02-23)
orders_enriched.parquet: 123,655 rows x 30 columns


All 123,655 rows find commission, discount, weekly fruit cost and pack+labor cost; no row is lost. Berry launch day uses the 2026-03-02 fruit cost (33.69 THB).

price / (1 - discount) equals base_price_thb on every row (0 rows differ), so list price = base_price_thb: 45 / 49 / 49 / 65 / 89 THB.

### 2.1 What does one cup earn, per item?

Mar-Aug 2026, all 5 items on sale. CM per cup = sell price - commission - fruit - pack+labor. CM % = CM / sell price (after discount, before commission).

In [14]:
MIX_START, MIX_END = "2026-03-01", "2026-08-31"
mix_period = enriched[enriched["date"].between(MIX_START, MIX_END)]
ue_item = unit_economics(mix_period, "sku").sort_values("cm_per_cup")
display(ue_item.round(2))

# One bar per item: the list price split into where each baht goes.
parts = {"CM": "cm_per_cup", "Fruit": "fruit_per_cup", "Pack + labor": "pack_labor_per_cup",
         "Commission": "commission_per_cup", "Discount": "discount_per_cup"}
part_colors = {"CM": "#2a78d6", "Fruit": "#eb6834", "Pack + labor": "#1baf7a",
               "Commission": "#eda100", "Discount": "#e87ba4"}
split = ue_item[list(parts.values())].set_axis(list(parts), axis=1)
assert np.allclose(split.sum(axis=1), ue_item["list_price"]), "the parts must add up to the list price"
plot = split.reset_index().melt(id_vars="sku", var_name="part", value_name="thb_per_cup")

low, high = ue_item.iloc[0], ue_item.iloc[-1]
fig = px.bar(plot, x="thb_per_cup", y="sku", color="part", orientation="h",
             category_orders={"sku": list(ue_item.index), "part": list(parts)}, color_discrete_map=part_colors,
             labels={"thb_per_cup": "THB per cup (adds up to the list price)", "sku": "", "part": ""})
fig.update_traces(marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{y}: %{x:.2f} THB<extra>%{fullData.name}</extra>")
fig.update_layout(height=420, barmode="stack", title=subtitle(
    f"{ue_item.index[0]} has the highest price ({low['list_price']:.0f} THB) but the lowest CM "
    f"({low['cm_per_cup']:.2f} THB a cup); fruit takes {low['fruit_per_cup']:.0f} THB of it",
    "Where each baht of the list price goes, per cup, Mar - Aug 2026"))
fig.show()

,cups,list_price,discount_per_cup,avg_sell_price,commission_per_cup,fruit_per_cup,pack_labor_per_cup,cm_per_cup,cm_pct,gross_revenue_thb,total_cm_thb
sku,,,,,,,,,,,
MixedBerryPremium,28044,89.0,2.03,86.97,23.67,38.28,15.0,10.02,11.52,2439126.88,281076.79
PassionFruit,43426,65.0,1.28,63.72,17.45,22.01,12.0,12.27,19.25,2767044.15,532663.85
Watermelon,131679,45.0,0.85,44.15,12.09,7.47,11.0,13.59,30.78,5813599.05,1789373.25
Guava,54138,49.0,0.92,48.08,13.22,9.62,11.5,13.73,28.56,2602686.45,743398.40
Pineapple,86855,49.0,0.87,48.13,13.14,8.91,11.0,15.09,31.35,4180505.56,1310552.97


**Berry earns the least per cup: 10.02 THB CM (12% of its 87 THB sell price).** Fruit is 38.28 THB a cup, 44% of the sell price; the other items pay 7-22 THB.

Pineapple earns the most, 15.09 THB a cup (31%). Watermelon earns 13.59 but sells 131,679 cups, so it brings the most CM: 1.79M THB.

### 2.2 How much less does a LINE MAN cup earn than a Grab cup?

In [15]:
ue_plat = unit_economics(mix_period, ["sku", "platform"]).reindex(ITEMS, level="sku")
display(ue_plat[["cups", "avg_sell_price", "discount_per_cup", "commission_per_cup", "cm_per_cup", "cm_pct"]].round(2))

cm_wide = ue_plat["cm_per_cup"].unstack("platform")
comm_wide = ue_plat["commission_per_cup"].unstack("platform")
gap = pd.DataFrame({"cm_gap_per_cup": cm_wide["Grab"] - cm_wide["LINE MAN"],
                    "commission_gap_per_cup": comm_wide["LINE MAN"] - comm_wide["Grab"]})
gap["other_gap"] = gap["cm_gap_per_cup"] - gap["commission_gap_per_cup"]
lm_cups = ue_plat.xs("LINE MAN", level="platform")["cups"]
gap["cm_lost_vs_grab_thb"] = gap["cm_gap_per_cup"] * lm_cups
display(gap.round(2))
print(f"If LINE MAN cups earned the Grab CM per cup: +{gap['cm_lost_vs_grab_thb'].sum():,.0f} THB CM in Mar-Aug")

plot = ue_plat.reset_index()
fig = px.bar(plot, x="sku", y="cm_per_cup", color="platform", barmode="group",
             category_orders={"sku": ITEMS, "platform": ["LINE MAN", "Grab"]},
             color_discrete_map={"LINE MAN": "#2a78d6", "Grab": "#eb6834"},
             text=plot["cm_per_cup"].map("{:.1f}".format),
             labels={"cm_per_cup": "CM per cup (THB)", "sku": "", "platform": ""})
fig.update_traces(textposition="outside", cliponaxis=False, marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{x}: %{y:.2f} THB a cup<extra>%{fullData.name}</extra>")
fig.update_layout(height=420, title=subtitle(
    f"A LINE MAN cup earns {gap['cm_gap_per_cup'].min():.1f}-{gap['cm_gap_per_cup'].max():.1f} THB less CM than a Grab cup; "
    f"berry on LINE MAN keeps only {cm_wide.loc['MixedBerryPremium', 'LINE MAN']:.1f} THB",
    "CM per cup by item and platform, Mar - Aug 2026 (commission LINE MAN 30%, Grab 24%)"))
fig.show()

cups  avg_sell_price  discount_per_cup  \
sku               platform                                            
Watermelon        Grab      57411           44.18              0.82   
                  LINE MAN  74268           44.13              0.87   
Pineapple         Grab      39146           48.24              0.76   
                  LINE MAN  47709           48.05              0.95   
Guava             Grab      22530           48.14              0.86   
                  LINE MAN  31608           48.03              0.97   
PassionFruit      Grab      18970           63.68              1.32   
                  LINE MAN  24456           63.75              1.25   
MixedBerryPremium Grab      12981           87.09              1.91   
                  LINE MAN  15063           86.88              2.12   

                            commission_per_cup  cm_per_cup  cm_pct  
sku               platform                                          
Watermelon        Grab                   10.60       15.08   34.13  
                  LINE MAN               13.24       12.44   28.18  
Pineapple         Grab                   11.58       16.62   34.45  
                  LINE MAN               14.41       13.84   28.80  
Guava             Grab                   11.55       15.40   31.99  
                  LINE MAN               14.41       12.54   26.11  
PassionFruit      Grab                   15.28       14.28   22.43  
                  LINE MAN               19.13       10.70   16.79  
MixedBerryPremium Grab                   20.90       12.98   14.90  
                  LINE MAN               26.06        7.48    8.61

,cm_gap_per_cup,commission_gap_per_cup,other_gap,cm_lost_vs_grab_thb
sku,,,,
Watermelon,2.64,2.63,0.01,196318.82
Pineapple,2.78,2.84,-0.06,132663.85
Guava,2.86,2.86,0.00,90309.49
PassionFruit,3.58,3.84,-0.26,87574.86
MixedBerryPremium,5.50,5.16,0.34,82827.13


If LINE MAN cups earned the Grab CM per cup: +589,694 THB CM in Mar-Aug


**A LINE MAN cup earns 2.6-5.5 THB less CM than a Grab cup; almost all of it is the 6-point commission gap.** Berry on LINE MAN keeps 7.48 THB vs 12.98 on Grab.

Sell prices are the same on both platforms (within 0.3 THB), so the gap is not from discounts. LINE MAN sells 56% of cups, so this costs ~0.6M THB CM in Mar-Aug vs Grab terms.

### 2.3 What is left per cup after waste?

Preview only: waste cost from `waste_daily` (fruit + packaging, filled days included) / cups sold, Mar-Aug 2026, per item. Waste has no platform, so it is not split by platform. Full P&L in lesson 8.

In [16]:
waste = tables["waste_daily"]
waste_mix = (waste[waste["date"].between(MIX_START, MIX_END)]
             .groupby("sku")[["units_wasted", "est_waste_cost_thb"]].sum())
after = ue_item[["cups", "cm_per_cup", "total_cm_thb"]].join(waste_mix)
after["waste_per_cup"] = after["est_waste_cost_thb"] / after["cups"]
after["cm_after_waste_per_cup"] = after["cm_per_cup"] - after["waste_per_cup"]
after["cm_after_waste_thb"] = after["total_cm_thb"] - after["est_waste_cost_thb"]
after["waste_rate_pct"] = after["units_wasted"] / (after["cups"] + after["units_wasted"]) * 100
after = after.sort_values("cm_after_waste_per_cup")
display(after.round(2))

worst = after.iloc[0]
fig = px.bar(after.reset_index(), x="cm_after_waste_per_cup", y="sku", orientation="h",
             color=np.where(after["cm_after_waste_per_cup"] < 0, "Loses money", "Earns money"),
             color_discrete_map={"Loses money": "#e34948", "Earns money": "#2a78d6"},
             text=after["cm_after_waste_per_cup"].map("{:.2f}".format).to_numpy(),
             category_orders={"sku": list(after.index)},
             labels={"cm_after_waste_per_cup": "CM after waste per cup (THB)", "sku": "", "color": ""})
fig.update_traces(textposition="outside", cliponaxis=False, marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{y}: %{x:.2f} THB a cup<extra></extra>")
fig.add_vline(x=0, line_color="#52514e", line_width=1)
fig.update_yaxes(ticklabelstandoff=10)
fig.update_layout(height=380, xaxis_range=[min(after["cm_after_waste_per_cup"].min(), 0) - 4,
                                           after["cm_after_waste_per_cup"].max() * 1.15],
                  title=subtitle(
                      f"After waste, {after.index[0]} loses {abs(worst['cm_after_waste_per_cup']):.2f} THB a cup; "
                      f"{worst['waste_rate_pct']:.0f}% of its prepared cups are thrown away",
                      "CM per cup minus waste cost per cup sold, Mar - Aug 2026"))
fig.show()

,cups,cm_per_cup,total_cm_thb,units_wasted,est_waste_cost_thb,waste_per_cup,cm_after_waste_per_cup,cm_after_waste_thb,waste_rate_pct
sku,,,,,,,,,
MixedBerryPremium,28044,10.02,281076.79,7251.00,334936.82,11.94,-1.92,-53860.03,20.54
PassionFruit,43426,12.27,532663.85,4195.00,117866.28,2.71,9.55,414797.57,8.81
Guava,54138,13.73,743398.40,3113.06,48612.77,0.90,12.83,694785.63,5.44
Watermelon,131679,13.59,1789373.25,4407.94,59434.58,0.45,13.14,1729938.67,3.24
Pineapple,86855,15.09,1310552.97,3908.00,58236.02,0.67,14.42,1252316.95,4.31


**After waste, berry loses 1.92 THB a cup: waste costs 11.94 THB per cup sold, more than its 10.02 CM.** 20.5% of prepared berry cups are thrown away, vs 3-9% for the others.

In Mar-Aug berry made 281k THB CM and wasted 335k THB. PassionFruit is next weakest (9.55 THB after waste, 8.8% waste rate).

### 2.4 Do the totals reconcile?

In [17]:
ue_all = unit_economics(enriched, "sku")
assert np.isclose(ue_all["total_cm_thb"].sum(), enriched["cm_thb"].sum())
assert np.isclose(ue_all["gross_revenue_thb"].sum(), TOTAL_REV), "enriched revenue != load_clean revenue"
assert ue_all["cups"].sum() == TOTAL_CUPS
assert np.isclose(ue_item["total_cm_thb"].sum(), mix_period["cm_thb"].sum())
assert np.isclose(ue_plat["total_cm_thb"].sum(), ue_item["total_cm_thb"].sum())
# Independent check from the raw inputs: price x cups x (1 - commission) - (fruit + pack+labor) x cups.
independent = (enriched["price_thb"] * enriched["units_sold"] * (1 - enriched["commission_rate"])
               - (enriched["fruit_cost_per_cup_thb"] + enriched["pack_labor_per_cup_thb"]) * enriched["units_sold"])
assert np.allclose(independent, enriched["cm_thb"])

bridge = enriched[MONEY_COLS].sum()
display(bridge.to_frame("full year THB").round(0))
print(f"Full year: revenue {TOTAL_REV:,.2f} THB = load_clean | CM {enriched['cm_thb'].sum():,.0f} THB "
      f"({enriched['cm_thb'].sum() / TOTAL_REV:.1%} of revenue)")

ue_out = pd.concat([
    ue_item.reset_index().assign(level="item", platform="All")
           .merge(after[["units_wasted", "est_waste_cost_thb", "waste_per_cup", "cm_after_waste_per_cup",
                         "waste_rate_pct"]].reset_index(), on="sku"),
    ue_plat.reset_index().assign(level="item_platform"),
], ignore_index=True)
ue_out.insert(0, "period", "2026-03 to 2026-08")
ue_out.to_csv(OUT / "unit_economics.csv", index=False, encoding="utf-8")
u = pd.read_csv(OUT / "unit_economics.csv")
assert np.isclose(u.loc[u["level"] == "item", "total_cm_thb"].sum(), u.loc[u["level"] == "item_platform", "total_cm_thb"].sum())
print(f"unit_economics.csv: {len(u)} rows ({(u['level'] == 'item').sum()} item, {(u['level'] == 'item_platform').sum()} item x platform)")

,full year THB
list_revenue_thb,31732479.0
discount_thb,585546.0
gross_revenue_thb,31146933.0
commission_thb,8517392.0
net_revenue_thb,22629541.0
fruit_cost_thb,7089224.0
pack_labor_thb,7043697.0
cm_thb,8496620.0


Full year: revenue 31,146,932.57 THB = load_clean | CM 8,496,620 THB (27.3% of revenue)
unit_economics.csv: 15 rows (5 item, 10 item x platform)


Revenue in the enriched data = load_clean: 31,146,932.57 THB. CM by item adds up to total CM; item x platform adds up to item. Each row's CM matches price x cups x (1 - commission) - costs, built from the raw inputs.

`outputs/unit_economics.csv` holds both levels; filter on `level` before summing.

### 2.5 Does fruit cost squeeze the margin?

In [18]:
weekly_cm = enriched.groupby(["sku", "week_start"], as_index=False).agg(
    cm_thb=("cm_thb", "sum"), units_sold=("units_sold", "sum"), fruit_cost_thb=("fruit_cost_thb", "sum"),
    days=("date", "nunique"))
assert np.isclose(weekly_cm["cm_thb"].sum(), enriched["cm_thb"].sum())
# Full weeks only: berry's launch week and the last week (31 Aug) hold 1 day each.
weekly_cm = weekly_cm[weekly_cm["days"] == 7].copy()
weekly_cm["cm_per_cup"] = weekly_cm["cm_thb"] / weekly_cm["units_sold"]
weekly_cm["fruit_per_cup"] = weekly_cm["fruit_cost_thb"] / weekly_cm["units_sold"]

worst = weekly_cm.loc[weekly_cm.groupby("sku")["cm_per_cup"].idxmin()].set_index("sku").reindex(ITEMS)
worst["best_cm_per_cup"] = weekly_cm.groupby("sku")["cm_per_cup"].max()
display(worst[["week_start", "cm_per_cup", "fruit_per_cup", "best_cm_per_cup"]]
        .round({"cm_per_cup": 2, "fruit_per_cup": 2, "best_cm_per_cup": 2}))

fig = px.line(weekly_cm, x="week_start", y="cm_per_cup", color="sku", category_orders={"sku": ITEMS},
              color_discrete_map=ITEM_COLORS,
              labels={"cm_per_cup": "CM per cup (THB)", "week_start": "Week starting", "sku": ""})
fig.update_traces(line_width=2, hovertemplate="Week of %{x|%d %b %Y}: %{y:.2f} THB a cup<extra>%{fullData.name}</extra>")
# Mark each item's worst week with a ring in the item's color.
fig.add_trace(go.Scatter(x=worst["week_start"], y=worst["cm_per_cup"], mode="markers+text", showlegend=False,
                         marker=dict(size=13, color="white", line=dict(width=2, color=[ITEM_COLORS[s] for s in worst.index])),
                         text=[f"{v:.1f}" for v in worst["cm_per_cup"]], textposition="bottom center",
                         hovertemplate="Worst week %{x|%d %b %Y}: %{y:.2f} THB<extra></extra>"))
fig.add_hline(y=0, line_color="#e34948", line_width=1)
berry_w = weekly_cm[weekly_cm["sku"] == "MixedBerryPremium"]
n_neg = int((berry_w["cm_per_cup"] < 0).sum())
fig.update_layout(height=460, hovermode="x unified", title=subtitle(
    f"Berry CM per cup is below 0 in {n_neg} of {len(berry_w)} full weeks (worst {worst.loc['MixedBerryPremium', 'cm_per_cup']:.2f} THB); "
    f"the 4 old items never go below {worst.loc[ITEMS[:4], 'cm_per_cup'].min():.1f} THB",
    "CM per cup by full week (before waste), all platforms. Rings = each item's worst week."))
fig.show()

,week_start,cm_per_cup,fruit_per_cup,best_cm_per_cup
sku,,,,
Watermelon,2026-06-01,11.51,8.36,14.92
Pineapple,2025-09-08,11.92,10.95,17.65
Guava,2026-07-06,11.45,10.56,16.03
PassionFruit,2026-07-06,3.69,29.19,18.19
MixedBerryPremium,2026-08-24,-5.15,54.01,15.39


In [19]:
weekly_plat = enriched.groupby(["sku", "platform", "week_start"], as_index=False).agg(
    cm_thb=("cm_thb", "sum"), units_sold=("units_sold", "sum"),
    fruit_cost_per_cup_thb=("fruit_cost_per_cup_thb", "first"), days=("date", "nunique"))
# Full weeks of the item, not of the platform: on some days one platform has no sales of an item.
item_days = enriched.groupby(["sku", "week_start"])["date"].nunique().rename("item_days")
weekly_plat = weekly_plat.join(item_days, on=["sku", "week_start"])
weekly_plat = weekly_plat[weekly_plat["item_days"] == 7].copy()
one_platform_days = (enriched.groupby(["sku", "date"])["platform"].nunique() == 1).groupby("sku").sum()
print("Days where an item sold on only one platform:", one_platform_days.reindex(ITEMS).to_dict())
weekly_plat["cm_per_cup"] = weekly_plat["cm_thb"] / weekly_plat["units_sold"]
neg = (weekly_plat.assign(neg=weekly_plat["cm_per_cup"] < 0)
       .groupby(["sku", "platform"])["neg"].agg(weeks_below_0="sum", full_weeks="size")
       .reindex(ITEMS, level="sku"))
display(neg)

berry_neg = weekly_plat[(weekly_plat["sku"] == "MixedBerryPremium") & (weekly_plat["cm_per_cup"] < 0)]
display(berry_neg[["platform", "week_start", "fruit_cost_per_cup_thb", "cm_per_cup"]]
        .round({"fruit_cost_per_cup_thb": 2, "cm_per_cup": 2}).reset_index(drop=True))

Days where an item sold on only one platform: {'Watermelon': 51, 'Pineapple': 48, 'Guava': 53, 'PassionFruit': 47, 'MixedBerryPremium': 21}


weeks_below_0  full_weeks
sku               platform                           
Watermelon        Grab                  0          52
                  LINE MAN              0          52
Pineapple         Grab                  0          52
                  LINE MAN              0          52
Guava             Grab                  0          52
                  LINE MAN              0          52
PassionFruit      Grab                  0          52
                  LINE MAN              0          52
MixedBerryPremium Grab                  2          26
                  LINE MAN              5          26

,platform,week_start,fruit_cost_per_cup_thb,cm_per_cup
0,Grab,2026-08-17,52.57,-0.71
1,Grab,2026-08-24,54.01,-2.62
2,LINE MAN,2026-07-06,44.87,-0.04
3,LINE MAN,2026-08-03,47.40,-4.76
4,LINE MAN,2026-08-10,50.92,-4.50
5,LINE MAN,2026-08-17,52.57,-6.64
6,LINE MAN,2026-08-24,54.01,-8.44


**Berry CM per cup is below 0 in 4 of its 26 full weeks (worst -5.15 THB, week of 24 Aug); on LINE MAN in 5 full weeks, on Grab in 2.** All are in Jul-Aug, when berry fruit costs 45-54 THB a cup (its range is 33-54).

The 4 old items never go below 0. PassionFruit dips to 3.69 THB (week of 6 Jul, fruit 29 THB); Watermelon, Pineapple and Guava stay above 11.4 THB. Fruit cost squeezes berry, not the menu.

? On 21-53 days per item, one platform has no sales of that item while the other carries the whole day. Totals are fine; check how the export splits orders by platform.

### 2.6 How sensitive are customers to price?

Price only changes through promo codes: 5 list prices x 4 codes = 20 price points. Elasticity = % change in cups for a 1% change in price.

In [20]:
price_points = enriched.groupby(["sku", "base_code"])["price_thb"].nunique()
# One price per item x code: 5 items x 4 codes = 20 price points (Pineapple and Guava share 49 THB).
assert (price_points == 1).all() and len(price_points) == 20
promo_days = (enriched[enriched["base_code"] == "RC101"].groupby("sku")["date"].nunique().reindex(ITEMS))
print(f"Price points (item x code): {len(price_points)} | RC101 promo days per item: {promo_days.to_dict()}")

import statsmodels.formula.api as smf


def fit_elasticity(rows):
    """log(cups) ~ log(price) + controls on item-hours. Returns HC1 and date-clustered results."""
    a = (rows.groupby(["date", "hour", "kitchen", "platform", "price_thb"], as_index=False)["units_sold"].sum())
    a["weekday"] = a["date"].dt.dayofweek
    a["month"] = a["date"].dt.strftime("%Y-%m")
    a["log_cups"] = np.log(a["units_sold"])
    a["log_price"] = np.log(a["price_thb"])
    formula = "log_cups ~ log_price + C(hour) + C(weekday) + C(kitchen) + C(platform) + C(month)"
    hc1 = smf.ols(formula, data=a).fit(cov_type="HC1")
    # Promos are set by day, so hours of one day are not independent. Clustered errors show this.
    clus = smf.ols(formula, data=a).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(a["date"])[0]})
    lo, hi = hc1.conf_int().loc["log_price"]
    clo, chi = clus.conf_int().loc["log_price"]
    return {"elasticity": hc1.params["log_price"], "ci_low": lo, "ci_high": hi, "p_value": hc1.pvalues["log_price"],
            "ci_low_by_day": clo, "ci_high_by_day": chi, "p_value_by_day": clus.pvalues["log_price"],
            "n": int(hc1.nobs), "promo_days": rows.loc[rows["base_code"] != "RC000", "date"].nunique()}


est = []
for s in ITEMS:
    item = enriched[enriched["sku"] == s]
    main = fit_elasticity(item[item["base_code"].isin(["RC000", "RC101"])])
    check = fit_elasticity(item)
    est.append({"sku": s, "estimate": "1 main: RC101 vs RC000", **main})
    est.append({"sku": s, "estimate": "2 check: all codes", **check})
elasticity = pd.DataFrame(est)
display(elasticity.set_index(["sku", "estimate"]).round(3))

Price points (item x code): 20 | RC101 promo days per item: {'Watermelon': 8, 'Pineapple': 3, 'Guava': 6, 'PassionFruit': 7, 'MixedBerryPremium': 8}


elasticity  ci_low  ci_high  \
sku               estimate                                              
Watermelon        1 main: RC101 vs RC000      -1.658  -1.888   -1.429   
                  2 check: all codes          -2.053  -2.155   -1.951   
Pineapple         1 main: RC101 vs RC000      -1.317  -1.785   -0.849   
                  2 check: all codes          -2.026  -2.148   -1.904   
Guava             1 main: RC101 vs RC000      -1.769  -2.139   -1.400   
                  2 check: all codes          -2.005  -2.137   -1.873   
PassionFruit      1 main: RC101 vs RC000      -1.241  -1.579   -0.903   
                  2 check: all codes          -1.993  -2.132   -1.854   
MixedBerryPremium 1 main: RC101 vs RC000      -1.137  -1.496   -0.778   
                  2 check: all codes          -1.821  -2.019   -1.623   

                                          p_value  ci_low_by_day  \
sku               estimate                                         
Watermelon        1 main: RC101 vs RC000      0.0         -1.914   
                  2 check: all codes          0.0         -2.269   
Pineapple         1 main: RC101 vs RC000      0.0         -2.121   
                  2 check: all codes          0.0         -2.294   
Guava             1 main: RC101 vs RC000      0.0         -2.598   
                  2 check: all codes          0.0         -2.294   
PassionFruit      1 main: RC101 vs RC000      0.0         -1.592   
                  2 check: all codes          0.0         -2.326   
MixedBerryPremium 1 main: RC101 vs RC000      0.0         -1.394   
                  2 check: all codes          0.0         -2.302   

                                          ci_high_by_day  p_value_by_day  \
sku               estimate                                                 
Watermelon        1 main: RC101 vs RC000          -1.403           0.000   
                  2 check: all codes              -1.838           0.000   
Pineapple         1 main: RC101 vs RC000          -0.513           0.001   
                  2 check: all codes              -1.757           0.000   
Guava             1 main: RC101 vs RC000          -0.941           0.000   
                  2 check: all codes              -1.715           0.000   
PassionFruit      1 main: RC101 vs RC000          -0.890           0.000   
                  2 check: all codes              -1.659           0.000   
MixedBerryPremium 1 main: RC101 vs RC000          -0.880           0.000   
                  2 check: all codes              -1.340           0.000   

                                              n  promo_days  
sku               estimate                                   
Watermelon        1 main: RC101 vs RC000  29015           8  
                  2 check: all codes      31117          33  
Pineapple         1 main: RC101 vs RC000  27445           3  
                  2 check: all codes      29441          28  
Guava             1 main: RC101 vs RC000  25163           6  
                  2 check: all codes      26973          31  
PassionFruit      1 main: RC101 vs RC000  23675           7  
                  2 check: all codes      25306          31  
MixedBerryPremium 1 main: RC101 vs RC000  10066           8  
                  2 check: all codes      10818          22

In [21]:
main = elasticity[elasticity["estimate"].str.startswith("1")].set_index("sku")
allc = elasticity[elasticity["estimate"].str.startswith("2")].set_index("sku")
fig = go.Figure()
for label, t, color, shift in [("Main: RC101 vs RC000", main, "#2a78d6", -0.12), ("Check: all codes", allc, "#eb6834", 0.12)]:
    y = np.arange(len(ITEMS)) + shift
    fig.add_trace(go.Scatter(x=t.loc[ITEMS, "elasticity"], y=y, mode="markers", name=label,
                             marker=dict(size=11, color=color),
                             error_x=dict(type="data", symmetric=False, color=color, thickness=2,
                                          array=t.loc[ITEMS, "ci_high_by_day"] - t.loc[ITEMS, "elasticity"],
                                          arrayminus=t.loc[ITEMS, "elasticity"] - t.loc[ITEMS, "ci_low_by_day"]),
                             hovertemplate="%{x:.2f}<extra>" + label + "</extra>"))
for v in [-0.5, -1.0, -1.5]:
    fig.add_vline(x=v, line_dash="dot", line_color="#b0aea6", line_width=1)
fig.update_yaxes(tickvals=list(range(len(ITEMS))), ticktext=ITEMS, autorange="reversed")
fig.update_xaxes(title="Price elasticity (cups % change for a 1% price change)")
fig.update_layout(height=420, title=subtitle(
    f"Main estimate: a 1% price cut sells {-main['elasticity'].max():.1f}-{-main['elasticity'].min():.1f}% more cups; "
    f"the all-code check is steeper ({allc['elasticity'].max():.1f} to {allc['elasticity'].min():.1f})",
    "Point = estimate, bar = 95% CI clustered by day. Dotted lines = -0.5 / -1.0 / -1.5 used in 2.7."))
fig.show()

In [22]:
# The promo covers a whole day, so the real sample is days. Check the effect at day level, and with fake promo days.
rng = np.random.default_rng(0)
checks = []
for s in ITEMS:
    item = enriched[enriched["sku"] == s]
    other_promo = item.loc[item["base_code"].isin(["RC102", "RC103"]), "date"].unique()
    days_s = (item[~item["date"].isin(other_promo)].groupby("date")
              .agg(cups=("units_sold", "sum"), promo=("base_code", lambda c: (c == "RC101").any())))
    controls = pd.get_dummies(pd.DataFrame({"weekday": days_s.index.dayofweek.astype(str),
                                            "month": days_s.index.strftime("%Y-%m")}), drop_first=True).astype(float).to_numpy()
    y = np.log(days_s["cups"].to_numpy())

    def coef(log_price):
        X = np.column_stack([np.ones(len(y)), log_price, controls])
        return np.linalg.lstsq(X, y, rcond=None)[0][1]

    real = coef(np.where(days_s["promo"], np.log(0.85), 0.0))
    normal_days = np.flatnonzero(~days_s["promo"].to_numpy())
    n_promo = int(days_s["promo"].sum())
    fake = []
    for _ in range(1000):
        lp = np.zeros(len(y))
        lp[rng.choice(normal_days, n_promo, replace=False)] = np.log(0.85)
        fake.append(coef(lp))
    fake = np.array(fake)
    checks.append({"sku": s, "day_elasticity": real, "promo_days": n_promo, "days": len(y),
                   "placebo_low": np.percentile(fake, 2.5), "placebo_high": np.percentile(fake, 97.5),
                   "placebo_p": np.mean(np.abs(fake) >= abs(real))})
day_check = pd.DataFrame(checks).set_index("sku")
display(day_check.round(3))

,day_elasticity,promo_days,days,placebo_low,placebo_high,placebo_p
sku,,,,,,
Watermelon,-1.808,8,340,-0.345,0.291,0.0
Pineapple,-1.602,3,340,-0.668,0.439,0.0
Guava,-2.132,6,340,-0.410,0.400,0.0
PassionFruit,-1.542,7,341,-0.403,0.392,0.0
MixedBerryPremium,-1.510,8,170,-0.609,0.765,0.0


Main estimate (RC101 vs RC000, item-hours): -1.1 (berry) to -1.8 (Guava). A 15% bundle cut sells ~20-33% more cups. None is positive.

Day-level check (cups per day, weekday + month controls): -1.5 to -2.1, steeper than the hour model. 1,000 fake promo runs on normal days stay within -0.7 to +0.8, so the promo effect is real (placebo p < 0.001). The all-code check is steeper again (-1.8 to -2.1): RC102 double dates and rainy-season RC103 mix price with date and season.

Limits: each item has only 3-8 whole promo days, so the day-clustered p-values (< 0.01) are optimistic and any day shock on a promo day is in the estimate. Promo also gets app visibility, so this can overstate price sensitivity. We only see price cuts, not increases.

### 2.7 What happens if we raise the list price?

+3 / +5 / +10 THB on the list price, elasticity -0.5 / -1.0 / -1.5 and the item's main estimate. Base = Mar-Aug 2026 average per month. Discount share, commission share and costs per cup stay the same; CM = sell price x (1 - commission) - fruit - pack+labor. Before waste.

In [23]:
N_MONTHS = 6  # Mar-Aug 2026
base = mix_period.groupby("sku")[["units_sold", "list_revenue_thb", "gross_revenue_thb", "commission_thb",
                                   "fruit_cost_thb", "pack_labor_thb", "cm_thb"]].sum().reindex(ITEMS)
base["cups_per_month"] = base["units_sold"] / N_MONTHS
base["list_price"] = base["list_revenue_thb"] / base["units_sold"]
base["sell_share"] = base["gross_revenue_thb"] / base["list_revenue_thb"]  # 1 - average discount
base["commission_share"] = base["commission_thb"] / base["gross_revenue_thb"]
base["cost_per_cup"] = (base["fruit_cost_thb"] + base["pack_labor_thb"]) / base["units_sold"]
base["cm_per_month"] = base["cm_thb"] / N_MONTHS


def cm_per_cup(row, list_price):
    return list_price * row["sell_share"] * (1 - row["commission_share"]) - row["cost_per_cup"]


assert np.allclose([cm_per_cup(r, r["list_price"]) * r["cups_per_month"] for _, r in base.iterrows()], base["cm_per_month"])

sims = []
for s, r in base.iterrows():
    cm0 = cm_per_cup(r, r["list_price"])
    own = main.loc[s, "elasticity"]
    for add in [3, 5, 10]:
        p1 = r["list_price"] + add
        cm1 = cm_per_cup(r, p1)
        # Elasticity at which the price rise gives the same CM per month as today.
        breakeven = np.log(cm0 / cm1) / np.log(p1 / r["list_price"])
        for label, el in [("-0.5", -0.5), ("-1.0", -1.0), ("-1.5", -1.5), ("own estimate", own)]:
            cups1 = r["cups_per_month"] * (p1 / r["list_price"]) ** el
            sims.append({"sku": s, "price_add_thb": add, "list_price_now": r["list_price"], "list_price_new": p1,
                         "elasticity_case": label, "elasticity": el, "cups_change_pct": (cups1 / r["cups_per_month"] - 1) * 100,
                         "cups_per_month_now": r["cups_per_month"], "cups_per_month_new": cups1,
                         "cm_per_cup_now": cm0, "cm_per_cup_new": cm1,
                         "cm_per_month_now": r["cm_per_month"], "cm_per_month_new": cups1 * cm1,
                         "cm_change_per_month": cups1 * cm1 - r["cm_per_month"], "breakeven_elasticity": breakeven})
price_simulation = pd.DataFrame(sims)

show = price_simulation.pivot_table(index=["sku", "price_add_thb"], columns="elasticity_case",
                                    values="cm_change_per_month", sort=False).round(0)
display(show)
display(price_simulation.drop_duplicates(["sku", "price_add_thb"])
        .set_index(["sku", "price_add_thb"])[["list_price_new", "cm_per_cup_now", "cm_per_cup_new", "breakeven_elasticity"]].round(2))

elasticity_case                      -0.5     -1.0     -1.5  own estimate
sku               price_add_thb                                          
Watermelon        3               35948.0  25337.0  15062.0       11875.0
                  5               58864.0  40539.0  23154.0       17833.0
                  10             112962.0  73707.0  38200.0       27670.0
Pineapple         3               23714.0  16626.0   9745.0       12240.0
                  5               38885.0  26684.0  15060.0       19250.0
                  10              74852.0  48845.0  25144.0       33566.0
Guava             3               15064.0  10996.0   7047.0        4968.0
                  5               24694.0  17647.0  10935.0        7454.0
                  10              47503.0  32304.0  18452.0       11509.0
PassionFruit      3               13132.0  10858.0   8636.0        9780.0
                  5               21595.0  17580.0  13711.0       15696.0
                  10              41835.0  32817.0  24420.0       28692.0
MixedBerryPremium 3                9039.0   8120.0   7217.0        7871.0
                  5               14911.0  13246.0  11626.0       12797.0
                  10              29091.0  25154.0  21421.0       24110.0

list_price_new  cm_per_cup_now  \
sku               price_add_thb                                   
Watermelon        3                        48.0           13.59   
                  5                        50.0           13.59   
                  10                       55.0           13.59   
Pineapple         3                        52.0           15.09   
                  5                        54.0           15.09   
                  10                       59.0           15.09   
Guava             3                        52.0           13.73   
                  5                        54.0           13.73   
                  10                       59.0           13.73   
PassionFruit      3                        68.0           12.27   
                  5                        70.0           12.27   
                  10                       75.0           12.27   
MixedBerryPremium 3                        92.0           10.02   
                  5                        94.0           10.02   
                  10                       99.0           10.02   

                                 cm_per_cup_new  breakeven_elasticity  
sku               price_add_thb                                        
Watermelon        3                       15.73                 -2.26  
                  5                       17.15                 -2.21  
                  10                      20.71                 -2.10  
Pineapple         3                       17.23                 -2.23  
                  5                       18.66                 -2.19  
                  10                      22.23                 -2.09  
Guava             3                       15.87                 -2.43  
                  5                       17.29                 -2.37  
                  10                      20.84                 -2.25  
PassionFruit      3                       14.40                 -3.56  
                  5                       15.83                 -3.44  
                  10                      19.38                 -3.20  
MixedBerryPremium 3                       12.16                 -5.82  
                  5                       13.58                 -5.56  
                  10                      17.14                 -5.04

In [24]:
plot = pd.concat([price_simulation,
                  price_simulation.drop_duplicates(["sku", "elasticity_case"]).assign(price_add_thb=0, cm_change_per_month=0.0)])
plot = plot.sort_values("price_add_thb")
# Short panel names: long ones overlap above the narrow panels.
SHORT = {"PassionFruit": "Passion fruit", "MixedBerryPremium": "Berry"}
plot["item"] = plot["sku"].map(lambda s: SHORT.get(s, s))
fig = px.line(plot, x="price_add_thb", y="cm_change_per_month", color="elasticity_case", facet_col="item",
              facet_col_spacing=0.035, markers=True,
              category_orders={"item": [SHORT.get(s, s) for s in ITEMS],
                               "elasticity_case": ["-0.5", "-1.0", "-1.5", "own estimate"]},
              color_discrete_map={"-0.5": "#2a78d6", "-1.0": "#1baf7a", "-1.5": "#eb6834", "own estimate": "#4a3aa7"},
              labels={"cm_change_per_month": "CM change per month (THB)", "price_add_thb": "List price + THB",
                      "elasticity_case": "Elasticity"})
fig.update_traces(line_width=2, marker_size=7,
                  hovertemplate="+%{x} THB: %{y:,.0f} THB a month<extra>%{fullData.name}</extra>")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.add_hline(y=0, line_color="#52514e", line_width=1)
fig.update_xaxes(tickvals=[0, 3, 5, 10], title_text="")
fig.add_annotation(text="List price + THB", x=0.45, y=-0.14, xref="paper", yref="paper", showarrow=False,
                   font_size=14)
be5 = price_simulation[price_simulation["price_add_thb"] == 5].drop_duplicates("sku").set_index("sku")["breakeven_elasticity"]
fig.update_layout(height=440, margin=dict(t=125, b=80), title=subtitle(
    f"Before waste, +5 THB raises CM for every item even at -1.5; CM only falls if elasticity is below "
    f"{be5.max():.1f} ({be5.idxmax()}) to {be5.min():.1f} ({be5.idxmin()})",
    "CM change per month vs today (Mar-Aug average), by list price rise and elasticity"))
fig.show()

At +5 THB every item gains CM in every case: +7k THB a month (Guava, own -1.8) to +59k (Watermelon, -0.5), while cups fall 3-16%.

**But the break-even elasticity is only -2.2 to -2.4 for Watermelon, Pineapple and Guava**: close to the all-code check (-2.0) and the day-level check (Guava -2.1). PassionFruit (-3.4) and berry (-5.6) have much more room.

Not in this simulation: waste (berry loses money after waste, 2.3), competitor prices and lost customers.

### 2.8 Are the pricing tables saved and consistent?

In [25]:
elasticity.to_csv(OUT / "elasticity.csv", index=False, encoding="utf-8")
price_simulation.to_csv(OUT / "price_simulation.csv", index=False, encoding="utf-8")

el = pd.read_csv(OUT / "elasticity.csv")
ps = pd.read_csv(OUT / "price_simulation.csv")
assert len(el) == 2 * len(ITEMS) and el["elasticity"].lt(0).all()
assert len(ps) == len(ITEMS) * 3 * 4
now = ps.drop_duplicates("sku").set_index("sku")["cm_per_month_now"].sum() * N_MONTHS
assert np.isclose(now, mix_period["cm_thb"].sum()), "simulation base != Mar-Aug CM"
print(f"elasticity.csv: {len(el)} rows | price_simulation.csv: {len(ps)} rows | base CM Mar-Aug {now:,.0f} THB")

elasticity.csv: 10 rows | price_simulation.csv: 60 rows | base CM Mar-Aug 4,657,065 THB


`elasticity.csv`: 2 estimates x 5 items. `price_simulation.csv`: 5 items x 3 price rises x 4 elasticities. The simulation base adds back to Mar-Aug CM (4.66M THB).

### 2.9 What is the pricing decision per item?

| Rule | Condition (in order) | Decision |
|---|---|---|
| R1 | Item CM after waste below -1 THB a cup in 2+ full weeks, or below 0 in the latest month | fix cost (price and cost actions, latest-month base) |
| R2 + R5 | Whole day-clustered CI and the day-level estimate above the break-even elasticity of the next menu step; CI width at most 1.0; p (by day) < 0.05; CM still up at -1.5 | raise one step |
| R3 | Whole CI below the break-even elasticity | keep price |
| R4 | Anything else (CI crosses break-even or is wider than 1.0) | keep + 4-week test in one kitchen |
| R6 | Platform prices | test only |

In [26]:
from src.load_clean import week_monday

STEP = {"Watermelon": 49, "Pineapple": 55, "Guava": 55, "PassionFruit": 69, "MixedBerryPremium": 99}
WIDE_CI = 1.0
WASTE_TARGET = 0.10  # waste rate after cutting prep
LATEST = "2026-08"   # latest month: berry demand is falling, so the Mar-Aug average is too high for it

# R1: item CM after waste per cup, full weeks only (waste has no platform).
waste_week = (tables["waste_daily"].assign(week_start=lambda d: week_monday(d["date"]))
              .groupby(["sku", "week_start"])["est_waste_cost_thb"].sum())
wk = (enriched.groupby(["sku", "week_start"])
      .agg(cm_thb=("cm_thb", "sum"), units_sold=("units_sold", "sum"), days=("date", "nunique"))
      .join(waste_week))
wk = wk[wk["days"] == 7]
wk["cm_after_waste_per_cup"] = (wk["cm_thb"] - wk["est_waste_cost_thb"]) / wk["units_sold"]
weeks_below_1 = (wk["cm_after_waste_per_cup"] < -1).groupby("sku").sum()

in_latest = enriched["date"].dt.strftime("%Y-%m") == LATEST
waste_latest = tables["waste_daily"][tables["waste_daily"]["date"].dt.strftime("%Y-%m") == LATEST]
latest = (enriched[in_latest].groupby("sku")[["units_sold", "list_revenue_thb", "gross_revenue_thb", "commission_thb",
                                              "fruit_cost_thb", "pack_labor_thb", "cm_thb"]].sum()
          .join(waste_latest.groupby("sku")[["units_wasted", "est_waste_cost_thb"]].sum()))
latest["cm_after_waste"] = latest["cm_thb"] - latest["est_waste_cost_thb"]
display(pd.DataFrame({"full_weeks_below_-1": weeks_below_1, f"cm_after_waste_{LATEST}": latest["cm_after_waste"]})
        .reindex(ITEMS).round(0))

# Break-even fruit cost per platform (Mar-Aug): sell price x (1 - commission) - pack+labor - waste per cup.
be = ue_plat[["avg_sell_price", "commission_per_cup", "pack_labor_per_cup"]].copy()
be["waste_per_cup"] = be.index.get_level_values("sku").map(after["waste_per_cup"])
be["breakeven_fruit"] = be["avg_sell_price"] - be["commission_per_cup"] - be["pack_labor_per_cup"] - be["waste_per_cup"]
fc = tables["weekly_fruit_cost"]
fc_mix = fc[fc["week_start"].between("2026-03-02", MIX_END)]
be["cost_weeks_fruit_above"] = [int((fc_mix.loc[fc_mix["sku"] == s, "fruit_cost_per_cup_thb"] > v).sum())
                                for (s, _), v in be["breakeven_fruit"].items()]
be["cost_weeks"] = [int((fc_mix["sku"] == s).sum()) for s, _ in be.index]
display(be.round(2))


def step_change(s, el, r=None):
    """CM change per month (before waste) if the list price moves to the next menu step."""
    r = base.loc[s] if r is None else r
    p1 = STEP[s]
    cups1 = r["cups_per_month"] * (p1 / r["list_price"]) ** el
    return cups1 * cm_per_cup(r, p1) - r["cm_per_month"]


def step_breakeven(s):
    r = base.loc[s]
    return np.log(cm_per_cup(r, r["list_price"]) / cm_per_cup(r, STEP[s])) / np.log(STEP[s] / r["list_price"])


def latest_base(s):
    """Same fields as `base`, from the latest month only."""
    r = latest.loc[s]
    out = pd.Series({"cups_per_month": r["units_sold"], "list_price": r["list_revenue_thb"] / r["units_sold"],
                     "sell_share": r["gross_revenue_thb"] / r["list_revenue_thb"],
                     "commission_share": r["commission_thb"] / r["gross_revenue_thb"],
                     "cost_per_cup": (r["fruit_cost_thb"] + r["pack_labor_thb"]) / r["units_sold"],
                     "cm_per_month": r["cm_thb"]})
    return out


RISK = {
    "Watermelon": "38% of cups; the Mar-Aug base includes the April peak, so Sep-Nov gains are smaller",
    "Pineapple": "break-even -2.16 is just below the CI low (-2.12); only 3 promo days",
    "Guava": "break-even -2.34 is inside the CI (-2.60 to -0.94)",
    "PassionFruit": "rainy-season fruit cost (up to 29.7 THB) is near the LINE MAN break-even (29.9)",
    "MixedBerryPremium": "volume still falling; after both actions still below 0 at August volume",
}
CONFIDENCE = {"Watermelon": "medium", "Pineapple": "low", "Guava": "low", "PassionFruit": "medium",
              "MixedBerryPremium": "direction high, size low"}

rows = []
for s in ITEMS:
    m = main.loc[s]
    lo, hi = m["ci_low_by_day"], m["ci_high_by_day"]
    own, day_el = m["elasticity"], day_check.loc[s, "day_elasticity"]
    be_el = step_breakeven(s)
    row = {"sku": s, "price_now": base.loc[s, "list_price"], "next_step": STEP[s], "elasticity": own,
           "ci_low": lo, "ci_high": hi, "day_elasticity": day_el, "breakeven_elasticity": be_el,
           "new_price": base.loc[s, "list_price"], "test_price": np.nan,
           "cm_change_low": np.nan, "cm_change_high": np.nan, "cm_change_if_test_low": np.nan, "cm_change_if_test_high": np.nan,
           "cost_action": "", "cost_action_thb": np.nan, "cm_after_both_low": np.nan, "cm_after_both_high": np.nan,
           "delist_thb": np.nan, "base_period": "2026-03 to 2026-08"}
    low = min(step_change(s, -1.5), step_change(s, own))
    high = step_change(s, -0.5)
    r1 = weeks_below_1.get(s, 0) >= 2 or latest.loc[s, "cm_after_waste"] < 0
    if r1:
        lb = latest_base(s)
        q, waste_now = lb["cups_per_month"], latest.loc[s, "est_waste_cost_thb"]
        cost_per_wasted = waste_now / latest.loc[s, "units_wasted"]
        changes, after_both = [], []
        for el in [-1.5, own, -0.5]:
            cups1 = q * (STEP[s] / lb["list_price"]) ** el
            cm1 = cups1 * cm_per_cup(lb, STEP[s])
            changes.append(cm1 - lb["cm_per_month"])
            after_both.append(cm1 - WASTE_TARGET / (1 - WASTE_TARGET) * cups1 * cost_per_wasted)
        # R5 also applies to a price action inside R1.
        assert changes[0] > 0, "price action fails R5"
        waste_saving = waste_now - WASTE_TARGET / (1 - WASTE_TARGET) * q * cost_per_wasted
        row.update(rule="R1", decision="fix cost + review delist" if max(after_both) < 0 else "fix cost",
                   new_price=STEP[s], cm_change_low=min(changes), cm_change_high=max(changes),
                   cost_action=f"cut waste to {WASTE_TARGET:.0%}", cost_action_thb=waste_saving,
                   cm_after_both_low=min(after_both), cm_after_both_high=max(after_both),
                   delist_thb=-latest.loc[s, "cm_after_waste"], base_period=LATEST)
    elif (lo > be_el and day_el > be_el and hi - lo <= WIDE_CI and m["p_value_by_day"] < 0.05
          and step_change(s, -1.5) > 0 and step_change(s, own) > 0):
        row.update(rule="R2+R5", decision="raise", new_price=STEP[s], cm_change_low=low, cm_change_high=high)
    elif hi < be_el:
        row.update(rule="R3", decision="keep price")
    else:
        row.update(rule="R4", decision="keep + test", test_price=STEP[s],
                   cm_change_if_test_low=low, cm_change_if_test_high=high)
    row.update(main_risk=RISK[s], confidence=CONFIDENCE[s])
    rows.append(row)

# R6: platform pricing is a test only.
bp = enriched[(enriched["sku"] == "MixedBerryPremium") & enriched["date"].between(MIX_START, MIX_END)]
bp = bp.groupby("platform")[["cm_thb", "units_sold"]].sum()
berry_plat_after = bp["cm_thb"] / bp["units_sold"] - after.loc["MixedBerryPremium", "waste_per_cup"]
rows.append({"sku": "MixedBerryPremium (LINE MAN)", "rule": "R6", "decision": "test only", "price_now": 89.0,
             "new_price": 89.0, "base_period": "2026-03 to 2026-08",
             "cost_action": (f"test a LINE MAN-only price; berry after waste: LINE MAN {berry_plat_after['LINE MAN']:.2f} "
                             f"vs Grab {berry_plat_after['Grab']:.2f} THB a cup"),
             "main_risk": "platform terms may require the same price on every app", "confidence": "low"})
price_recommendation = pd.DataFrame(rows)

expected = {"Watermelon": "raise", "Pineapple": "keep + test", "Guava": "keep + test",
            "PassionFruit": "raise", "MixedBerryPremium": "fix cost + review delist"}
assert price_recommendation.set_index("sku")["decision"].reindex(list(expected)).to_dict() == expected, "rules changed the decision"

show_cols = ["sku", "rule", "decision", "price_now", "new_price", "test_price", "breakeven_elasticity", "ci_low", "ci_high",
             "cm_change_low", "cm_change_high", "cm_change_if_test_low", "cm_change_if_test_high",
             "cost_action_thb", "cm_after_both_low", "cm_after_both_high", "delist_thb", "base_period"]
display(price_recommendation[show_cols].round(2))
b = price_recommendation.set_index("sku").loc["MixedBerryPremium"]
need = -b["cm_after_both_high"] / (latest.loc["MixedBerryPremium", "units_sold"] * (99 / 89) ** main.loc["MixedBerryPremium", "elasticity"])
print(f"Berry {LATEST}: CM after waste {latest.loc['MixedBerryPremium', 'cm_after_waste']:,.0f} THB | "
      f"after raise + waste cut: {b['cm_after_both_low']:,.0f} to {b['cm_after_both_high']:,.0f} THB a month | "
      f"needs ~{need:.1f} THB a cup cheaper fruit to break even")

price_recommendation.to_csv(OUT / "price_recommendation.csv", index=False, encoding="utf-8")
pr = pd.read_csv(OUT / "price_recommendation.csv")
assert len(pr) == len(ITEMS) + 1
print(f"price_recommendation.csv: {len(pr)} rows")

,full_weeks_below_-1,cm_after_waste_2026-08
sku,,
Watermelon,0,218545.0
Pineapple,0,144408.0
Guava,0,87413.0
PassionFruit,0,16472.0
MixedBerryPremium,14,-43328.0


avg_sell_price  commission_per_cup  \
sku               platform                                       
Watermelon        Grab               44.18               10.60   
                  LINE MAN           44.13               13.24   
Pineapple         Grab               48.24               11.58   
                  LINE MAN           48.05               14.41   
Guava             Grab               48.14               11.55   
                  LINE MAN           48.03               14.41   
PassionFruit      Grab               63.68               15.28   
                  LINE MAN           63.75               19.13   
MixedBerryPremium Grab               87.09               20.90   
                  LINE MAN           86.88               26.06   

                            pack_labor_per_cup  waste_per_cup  \
sku               platform                                      
Watermelon        Grab                    11.0           0.45   
                  LINE MAN                11.0           0.45   
Pineapple         Grab                    11.0           0.67   
                  LINE MAN                11.0           0.67   
Guava             Grab                    11.5           0.90   
                  LINE MAN                11.5           0.90   
PassionFruit      Grab                    12.0           2.71   
                  LINE MAN                12.0           2.71   
MixedBerryPremium Grab                    15.0          11.94   
                  LINE MAN                15.0          11.94   

                            breakeven_fruit  cost_weeks_fruit_above  \
sku               platform                                            
Watermelon        Grab                22.13                       0   
                  LINE MAN            19.44                       0   
Pineapple         Grab                24.99                       0   
                  LINE MAN            21.96                       0   
Guava             Grab                24.19                       0   
                  LINE MAN            21.22                       0   
PassionFruit      Grab                33.68                       0   
                  LINE MAN            29.91                       0   
MixedBerryPremium Grab                39.24                      14   
                  LINE MAN            33.87                      24   

                            cost_weeks  
sku               platform              
Watermelon        Grab              27  
                  LINE MAN          27  
Pineapple         Grab              27  
                  LINE MAN          27  
Guava             Grab              27  
                  LINE MAN          27  
PassionFruit      Grab              27  
                  LINE MAN          27  
MixedBerryPremium Grab              27  
                  LINE MAN          27

,sku,rule,decision,price_now,new_price,test_price,breakeven_elasticity,ci_low,ci_high,cm_change_low,cm_change_high,cm_change_if_test_low,cm_change_if_test_high,cost_action_thb,cm_after_both_low,cm_after_both_high,delist_thb,base_period
0,Watermelon,R2+R5,raise,45.0,49.0,NaN,-2.24,-1.91,-1.40,15025.96,47504.93,NaN,NaN,NaN,NaN,NaN,NaN,2026-03 to 2026-08
1,Pineapple,R4,keep + test,49.0,49.0,55.0,-2.16,-2.12,-0.51,NaN,NaN,17415.73,46294.24,NaN,NaN,NaN,NaN,2026-03 to 2026-08
2,Guava,R4,keep + test,49.0,49.0,55.0,-2.34,-2.60,-0.94,NaN,NaN,8489.83,29394.80,NaN,NaN,NaN,NaN,2026-03 to 2026-08
3,PassionFruit,R2+R5,raise,65.0,69.0,NaN,-3.50,-1.59,-0.89,11236.65,17391.36,NaN,NaN,NaN,NaN,NaN,NaN,2026-03 to 2026-08
4,MixedBerryPremium,R1,fix cost + review delist,89.0,99.0,NaN,-5.04,-1.39,-0.88,16273.12,17389.96,NaN,NaN,20487.72,-4594.64,-4130.53,43328.2,2026-08
5,MixedBerryPremium (LINE MAN),R6,test only,89.0,89.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03 to 2026-08


Berry 2026-08: CM after waste -43,328 THB | after raise + waste cut: -4,595 to -4,131 THB a month | needs ~1.8 THB a cup cheaper fruit to break even
price_recommendation.csv: 6 rows


**Berry: fix cost + review delist (R1).** At August volume it loses 43k THB a month after waste; raising to 99 and cutting waste to 10% still leave about -4k a month, and it needs ~1.8 THB a cup cheaper fruit to break even. Delisting would add +43k a month if customers do not switch.

Watermelon (45 -> 49) and PassionFruit (65 -> 69) are raised: their whole CI and the day-level estimate are above the break-even (-2.24, -3.50). Pineapple and Guava keep 49 with a 4-week test at 55. `price_recommendation.csv`: do not sum rows; R4 ranges are for the test price and berry uses an August base.

## Findings -> next steps (pricing)
1. Berry at August volume (82 cups a day) loses 43k THB a month after waste; raise to 99 (+16-17k) and waste to 10% (+20k) still leave about -4k -> fix cost and review delist; it needs ~1.8 THB a cup cheaper fruit (lesson 10).
2. A LINE MAN cup earns 2.6-5.5 THB less CM than a Grab cup (6-point commission), ~590k THB in Mar-Aug -> test a LINE MAN-only berry price, only if platform terms allow.
3. Watermelon and PassionFruit: whole CI above break-even (-2.24, -3.50) -> raise 45 -> 49 (+15-48k THB a month) and 65 -> 69 (+11-17k), Mar-Aug base, before waste.
4. Pineapple and Guava: CI wider than 1.0 or crossing break-even -> keep 49 and run a 4-week test at 55 in one kitchen (+8k to +46k a month if it holds).
5. Elasticity rests on 3-8 whole promo days; the day-level check is steeper (-1.5 to -2.1) and the placebo confirms the effect -> re-estimate after the price tests.